<p class="lead">Este notebook es la edición en R de la primera unidad del curso. Las diapositivas de clase son un mapa; este texto es el territorio. Está escrito para leerse de corrido una vez, ejecutando cada bloque de código a medida que llegas a él, y luego para consultarse por secciones. A diferencia de un capítulo de libro de texto, cada figura y cada número que contiene son producidos por código R que puedes ver y modificar: cuando una frase dice "la tabla muestra 444 casos benignos y 239 malignos", la tabla de arriba se calculó frente a ti. La unidad tiene dos mitades. La primera (Secciones 1&ndash;3) responde <em>qué es la inteligencia artificial</em>, de dónde proviene y qué familias de métodos comprende. La segunda (Secciones 4&ndash;7) introduce el <em>reconocimiento de patrones</em> &mdash; la disciplina a la que pertenece la mayor parte de este curso &mdash; y la pregunta de cómo deben manejar los datos sobre personas los ingenieros que construyen sistemas que aprenden de ellos.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Enunciar al menos dos definiciones operacionales de inteligencia artificial y explicar por qué el campo no tiene una única definición consensuada.</li>
<li>Ubicar los principales hitos de la historia de la IA en una línea de tiempo, nombrar a las personas asociadas a ellos y explicar qué causó los dos "inviernos de la IA".</li>
<li>Distinguir los enfoques simbólico, conexionista, estadístico y evolutivo de la IA y dar un ejemplo médico de cada uno; distinguir sistema, algoritmo, modelo y ecuación; clasificar un sistema como estrecho/general y como reactivo/de memoria limitada.</li>
<li>Describir qué es la IA generativa, qué modalidades existen y qué riesgos agrega a los de los sistemas de clasificación.</li>
<li>Definir el reconocimiento de patrones y describir su pipeline: sensado, representación, extracción de características, clasificación, decisión.</li>
<li>Representar un paciente, una imagen o una señal como un vector de características; clasificar tipos de datos (continuos, discretos, nominales, ordinales, binarios) y explicar por qué el tipo determina las operaciones admisibles.</li>
<li>Explicar qué es una muestra de entrenamiento, por qué es una muestra y no la población, y por qué esa distinción es la raíz de todos los modos de falla que se estudiarán más adelante en el curso.</li>
<li>Explicar la reidentificación, el k-anonimato, el cambio en la distribución de los datos (dataset shift) y el sesgo de etiquetas, y nombrar las regulaciones chilenas e internacionales que rigen los datos de salud.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código (chunk) es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que interpreta esa salida. Los ejercicios (cajas verdes) te piden modificar el bloque que está sobre ellos y volver a ejecutarlo; son la forma más rápida de aprender. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan esta unidad con otras unidades o con el curso de bioinformática (BGM324). El notebook usa solo R base, el paquete <code>stats</code> y los paquetes de modelado <code>mlbench</code>, <code>rpart</code>, <code>rpart.plot</code> y <code>class</code>, todos preinstalados en este entorno e instalables con una sola línea en un entorno de ejecución de R nuevo de Google Colab (ver más abajo); no se descargan datos, porque los conjuntos de datos vienen incluidos en el paquete <code>mlbench</code><sup class="c"><a href="#r38">[38]</a></sup> y en el propio R.<sup class="c"><a href="#r39">[39]</a></sup> El Suplemento S0 cubre las herramientas de R si no has usado estas bibliotecas antes.</p></div>



In [ ]:
# Configuración: bibliotecas usadas a lo largo del notebook. La semilla aleatoria se fija
# para que todo número citado en el texto sea reproducible. En un entorno R nuevo de Colab, instala una vez:
#   install.packages(c("mlbench","rpart","rpart.plot","class"))
suppressPackageStartupMessages({
  library(mlbench); library(rpart); library(rpart.plot); library(class)
})
set.seed(0)
cat(R.version.string, "| mlbench", as.character(packageVersion("mlbench")),
    "| rpart", as.character(packageVersion("rpart")), "\n")



La salida impresa registra la versión de R y las versiones de los paquetes con que se construyeron estos apuntes. `set.seed(0)` hace reproducible cada partición aleatoria posterior; cuando un bloque de código genera números aleatorios, vuelve a fijar la semilla explícitamente, de modo que el número citado en el texto que lo rodea sea el número que produce el bloque.

## 1. ¿Qué es la inteligencia artificial?

El término se acuñó en una propuesta de financiamiento. En el verano de 1955, John McCarthy, Marvin Minsky, Nathaniel Rochester y Claude Shannon solicitaron a la Fundación Rockefeller apoyo para un taller de dos meses en el Dartmouth College destinado a estudiar "la conjetura de que todo aspecto del aprendizaje o cualquier otro rasgo de la inteligencia puede, en principio, describirse con tal precisión que se pueda construir una máquina que lo simule".<sup class="c"><a href="#r1">[1]</a></sup> Esa frase contiene todo el programa del campo y también su ambigüedad central: define la inteligencia artificial por referencia a la <em>inteligencia</em>, que a su vez no tiene una definición consensuada.

Cinco años antes, Alan Turing había esquivado el problema. En lugar de preguntar "¿pueden pensar las máquinas?", una pregunta que consideraba "demasiado carente de sentido como para merecer discusión", propuso una prueba operacional: una máquina es inteligente si un interrogador que intercambia mensajes escritos con ella no puede distinguirla de manera confiable de un ser humano.<sup class="c"><a href="#r2">[2]</a></sup> El juego de imitación &mdash; la prueba de Turing &mdash; sigue siendo la definición de IA más conocida fuera del campo, y sigue siendo criticada dentro de él, porque mide la capacidad de <em>parecer</em> humano y no la capacidad de resolver problemas.

### 1.1 Cuatro maneras de definir el campo

Russell y Norvig, cuyo libro de texto ha sido la referencia estándar durante tres décadas, organizan las definiciones históricas a lo largo de dos ejes: si el objetivo es reproducir el comportamiento <em>humano</em> o lograr un comportamiento <em>racional</em> (ideal), y si el objeto de estudio es el <em>pensamiento</em> o la <em>acción</em>.<sup class="c"><a href="#r3">[3]</a></sup>

| | Pensar | Actuar |
|---|---|---|
| **Como humanos** | Modelado cognitivo: construir sistemas que piensen como lo hacen las personas (ciencia cognitiva, neurociencia) | La prueba de Turing: construir sistemas cuyo comportamiento sea indistinguible del de una persona |
| **Racionalmente** | Leyes del pensamiento: lógica formal, inferencia correcta a partir de premisas | **Agentes racionales**: construir sistemas que perciban su entorno y actúen de modo de maximizar el logro esperado de sus objetivos |

La IA moderna se ubica casi por completo en la celda inferior derecha. Un <strong>agente racional</strong> es cualquier cosa que percibe su entorno mediante sensores y actúa sobre él mediante actuadores para hacerlo lo mejor posible, dado lo que sabe. Un sistema de diagnóstico que lee una fotografía de retina y entrega una probabilidad de retinopatía diabética es un agente racional exactamente en este sentido: su "sensor" es la imagen, su "actuador" es el informe, y "hacerlo bien" se define mediante un objetivo medible (la concordancia con oftalmólogos). Esta definición tiene dos virtudes para los ingenieros: no exige que el sistema piense como una persona y hace explícito el objetivo, de modo que el éxito se puede medir.

### 1.2 Cuatro palabras que suelen confundirse: sistema, algoritmo, modelo, ecuación

La clase insiste en el vocabulario porque el uso impreciso de estas palabras es fuente de mucha confusión en la literatura y en la prensa.

- Un **sistema computacional** es la combinación de hardware, software y dispositivos periféricos que realiza una tarea. "La IA" que lee una radiografía es un sistema: una GPU, un sistema operativo, un programa, una conexión de red al archivo de imágenes del hospital.
- Un **algoritmo** es una secuencia finita y no ambigua de pasos que transforma una entrada en una salida. Needleman&ndash;Wunsch (BGM324), el descenso de gradiente (Unidad 2) y los k vecinos más cercanos (Unidad 3) son algoritmos. Un algoritmo no tiene parámetros aprendidos de los datos; es un procedimiento.
- Un **modelo** es un objeto con parámetros ajustables que asigna una salida a una entrada. Una regresión logística ajustada, un árbol de decisión entrenado y una red neuronal con sus pesos son modelos. Un *algoritmo de aprendizaje* produce un modelo a partir de datos; lo que se despliega es el modelo, no el algoritmo. Cuando las noticias dicen "el algoritmo decidió", casi siempre quieren decir "el modelo predijo".
- Una **ecuación** es el enunciado matemático de una relación; un *modelo matemático* es una o más ecuaciones tomadas como descripción de un fenómeno. La regla del centroide más cercano de la Sección 5 es una ecuación; una vez que sus dos centroides se completan a partir de datos, se convierte en un modelo.

### 1.3 Qué debe hacer un sistema para que se le llame inteligente

En términos operacionales, la clase enumera seis capacidades que los sistemas de IA exhiben en algún grado, y la lista es una útil guía de verificación para leer cualquier afirmación sobre un sistema "inteligente": **razonamiento y resolución de problemas** (analizar datos, identificar patrones, extraer conclusiones); **aprendizaje** (mejorar el desempeño a partir de datos y experiencia &mdash; el tema de las Unidades 2&ndash;8); **percepción** (interpretar entradas visuales, acústicas o lingüísticas &mdash; visión por computador, reconocimiento de voz, procesamiento de lenguaje natural); **toma de decisiones** (elegir una acción a partir de entradas, reglas o patrones aprendidos, normalmente optimizando un objetivo); **automatización** (realizar la tarea sin un humano en el ciclo); y **adaptabilidad** (cambiar el comportamiento cuando cambia el entorno). Un filtro de spam tiene aprendizaje, percepción de texto y toma de decisiones, pero no adaptabilidad a menos que se reentrene; un programa de ajedrez tiene razonamiento y toma de decisiones, pero no percepción; AlphaFold tiene aprendizaje y percepción de secuencias. Ningún sistema actual tiene las seis en el sentido general en que las tiene una persona, que es el contenido de la siguiente distinción.

### 1.4 IA estrecha, general y superinteligencia; los cuatro tipos funcionales

Los sistemas que existen hoy son instancias de <strong>IA estrecha (débil)</strong>: cada uno resuelve una tarea &mdash; clasificar imágenes, transcribir voz, predecir la estructura de proteínas, jugar Go &mdash; y no tiene competencia fuera de ella. La <strong>inteligencia artificial general (AGI)</strong>, un sistema con competencia de nivel humano en todas las tareas, no existe, y si los métodos actuales conducen a ella es una pregunta científica y filosófica abierta que este curso no necesita zanjar. Cuando un artículo de prensa dice "una IA aprendió a diagnosticar cáncer de piel", el sujeto es un sistema estrecho entrenado para un problema de clasificación, y la pregunta correcta es la que enseña este curso: <em>¿con qué datos, medido cómo, y sigue funcionando en pacientes distintos de aquellos con los que fue entrenado?</em>

La palabra <em>inteligencia</em> también se usa para varias capacidades humanas distintas. La psicología distingue, entre otras, la inteligencia <strong>fluida</strong> (razonar en situaciones nuevas) de la inteligencia <strong>cristalizada</strong> (conocimiento acumulado), y la influyente aunque discutida teoría de Gardner enumera las inteligencias lingüística, lógico&ndash;matemática, espacial, musical, corporal&ndash;cinestésica, interpersonal, intrapersonal y naturalista. La correspondencia con las máquinas es instructiva: los sistemas actuales son muy fuertes en tareas cristalizadas y lógico&ndash;matemáticas, y cada vez más en tareas lingüísticas y espaciales, y están prácticamente ausentes en las dimensiones interpersonal, intrapersonal y corporal. Un sistema que responde preguntas de exámenes médicos por encima del umbral de aprobación<sup class="c"><a href="#r4">[4]</a></sup> y un sistema que no puede tomar un lápiz de una mesa son el mismo tipo de cosa.

La divulgación agrega un tercer nivel, la **superinteligencia artificial (ASI)**, un sistema hipotético que supera la capacidad humana en todos los dominios; es un tema de la filosofía y de las políticas públicas, no de la ingeniería, y este curso lo menciona solo para que reconozcas el término. Una segunda clasificación, *funcional*, popularizada por Hintze<sup class="c"><a href="#r30">[30]</a></sup>, ordena los sistemas según lo que pueden hacer con el pasado:

| Tipo | Qué hace | Estado | Ejemplo |
|---|---|---|---|
| **Reactivo** | Responde a la entrada presente sin memoria del pasado; su comportamiento queda fijo una vez construido | Existe | Deep Blue de IBM (1997): evalúa el tablero actual, no recuerda ninguna partida anterior |
| **Memoria limitada** | Aprende de datos pasados almacenados y usa la experiencia reciente para decidir; aquí se ubican todos los sistemas de aprendizaje automático de este curso | Existe | Un clasificador entrenado; un auto autónomo que sigue a los vehículos cercanos durante los últimos segundos |
| **Teoría de la mente** | Modelaría las creencias, intenciones y emociones de los agentes con los que interactúa | Conceptual; en investigación | &mdash; |
| **Autoconsciente** | Poseería una representación de su propio estado interno | Hipotético | &mdash; |

La clase te pide reflexionar sobre los riesgos que suelen asociarse a los dos últimos niveles &mdash; comportamiento impredecible, automejora rápida ("explosión de inteligencia"), manipulación, ausencia de empatía o de juicio moral. La Sección 7 adopta una posición distinta y, para un ingeniero, más urgente: los sistemas que *existen* (los sistemas de memoria limitada) ya producen daño mediante mecanismos que se comprenden por completo &mdash; reidentificación, aprendizaje por atajos (shortcut learning), sesgo de etiquetas, desinformación &mdash; y prevenirlos es hoy una responsabilidad profesional, crea uno lo que crea sobre los niveles hipotéticos.

<div class="box warn"><h4>Error frecuente</h4><p>"Inteligencia artificial" y "aprendizaje automático" no son sinónimos. El aprendizaje automático (Unidad 2) es el subcampo de la IA en el que el comportamiento del sistema se <em>aprende a partir de datos</em> en lugar de programarse. Buena parte de la IA que operaba en hospitales en los años 1980 &mdash; los sistemas expertos &mdash; no contenía aprendizaje alguno, y buena parte de lo que hoy se comercializa como IA es aprendizaje automático. El curso se titula "Inteligencia Artificial" y su contenido es mayoritariamente aprendizaje automático, porque es allí donde se ha concentrado el impacto práctico del campo desde la década de 2010; la Sección 3 explica por qué.</p></div>

## 2. Una breve historia, con sus dos inviernos

La historia del campo es un ciclo de entusiasmo, promesas excesivas, decepción y recuperación. Conocerla te protege tanto de la exageración como de la reacción adversa del ciclo actual. La figura de abajo ubica los principales hitos en una línea de tiempo; los párrafos que siguen los explican.



In [ ]:
# Línea de tiempo de la inteligencia artificial: los hitos discutidos en la Sección 2.
events <- data.frame(
  year = c(1943,1950,1956,1958,1966,1969,1976,1986,1997,2011,2012,2016,2017,2020,2022),
  lab  = c("McCulloch & Pitts\nartificial neuron","Turing, 'Computing\nMachinery and Intelligence'",
           "Logic Theorist;\nDartmouth: the name 'AI'","Rosenblatt,\nperceptron","ELIZA",
           "Minsky & Papert,\n'Perceptrons'","MYCIN (medical\nexpert system)",
           "Backpropagation\n(Rumelhart, Hinton,\nWilliams)","Deep Blue beats\nKasparov",
           "IBM Watson\nwins Jeopardy!","AlexNet wins\nImageNet","AlphaGo beats\nLee Sedol",
           "Transformer\narchitecture","AlphaFold2","ChatGPT"),
  lev  = c(1,-1,2,-2,-2,1,-1,1,-1,-2,1,-1,2,-2,1))
winters <- data.frame(a=c(1974,1987), b=c(1980,1993), lab=c("first AI winter","second AI winter"))
h <- c(`-2`=-1.25, `-1`=-0.5, `1`=0.5, `2`=1.25)
par(mar=c(3,1,3,1))
plot(NA, xlim=c(1938,2028), ylim=c(-2.1,2.15), axes=FALSE, xlab="", ylab="",
     main="Milestones of artificial intelligence, with the two 'winters' of reduced funding shaded")
for(i in 1:nrow(winters)){ rect(winters$a[i],-2.1,winters$b[i],2.15,col="#dfe7f1",border=NA)
  text((winters$a[i]+winters$b[i])/2,1.95,winters$lab[i],col="#2b5d9b",font=3,cex=.75) }
abline(h=0, lwd=1)
for(i in 1:nrow(events)){ hi <- h[as.character(events$lev[i])]
  segments(events$year[i],0,events$year[i],hi,col="#2b5d9b",lwd=.9)
  points(events$year[i],0,pch=19,col="#d2691e",cex=.9)
  text(events$year[i], hi+ifelse(hi>0,.07,-.07), events$lab[i],
       adj=c(.5,ifelse(hi>0,0,1)), cex=.62) }
axis(1, at=seq(1940,2020,10), pos=-2.1)



<p class="caption"><b>Figura 1.</b> Línea de tiempo de los hitos discutidos en esta sección. Las bandas sombreadas son los dos períodos de reducción del financiamiento y del interés público. Observa la distancia entre el perceptrón (1958) y el método que finalmente hizo entrenables las redes multicapa (1986), y entre ese método y el hardware que lo hizo práctico a gran escala (2012).</p>

### 2.1 Fundamentos (1943&ndash;1956)

En 1943, el neurofisiólogo Warren McCulloch y el lógico Walter Pitts demostraron que un modelo simplificado de neurona &mdash; una unidad que suma entradas ponderadas y se activa si la suma supera un umbral &mdash; puede computar cualquier función lógica cuando las unidades se conectan en redes.<sup class="c"><a href="#r5">[5]</a></sup> Este es el antecesor de toda red neuronal artificial (Unidad 6). En 1949, el psicólogo Donald Hebb propuso que la conexión entre dos neuronas se refuerza cuando estas se activan juntas &mdash; el <strong>aprendizaje hebbiano</strong>, la primera regla para modificar los pesos de las conexiones a partir de la experiencia y la semilla conceptual de todo algoritmo de entrenamiento.<sup class="c"><a href="#r31">[31]</a></sup> El artículo de Turing de 1950 aportó el marco filosófico y, notablemente, anticipó el aprendizaje automático: en lugar de programar una mente adulta, sugirió, "¿por qué no intentar más bien producir una que simule la del niño?" y luego educarla.<sup class="c"><a href="#r2">[2]</a></sup> En 1955&ndash;56, Allen Newell y Herbert Simon escribieron el <strong>Logic Theorist</strong>, habitualmente considerado el primer programa de inteligencia artificial: demostró 38 de los primeros 52 teoremas de los <em>Principia Mathematica</em> de Whitehead y Russell, y encontró una demostración más corta para uno de ellos.<sup class="c"><a href="#r32">[32]</a></sup> El taller de Dartmouth de 1956 le dio al campo su nombre y su primera generación de líderes: McCarthy, Minsky, Newell y Simon.<sup class="c"><a href="#r1">[1]</a></sup> Esos mismos años produjeron los primeros lenguajes de programación de alto nivel &mdash; FORTRAN (1957), LISP (1958, creado por McCarthy para la IA) y COBOL (1959) &mdash;, sin los cuales nada de esto se podría haber escrito.

### 2.2 La primera ola y el primer invierno (1956&ndash;1980)

Los primeros años produjeron programas que demostraban teoremas, resolvían problemas de álgebra planteados en palabras y jugaban damas, y generaron predicciones que resultaron muy equivocadas (Simon en 1965: en veinte años las máquinas podrían hacer cualquier trabajo que pueda hacer un hombre). Dos artefactos del período todavía se citan. <strong>ELIZA</strong> (Joseph Weizenbaum, 1966), el primer chatbot, imitaba a un psicoterapeuta rogeriano devolviendo las frases del usuario en forma de preguntas; no entendía nada, y a Weizenbaum le alarmó la facilidad con que los usuarios se confiaban a él &mdash; el primer caso documentado de personas que atribuyen comprensión a un sistema que no la tiene, fenómeno que hoy se denomina efecto ELIZA.<sup class="c"><a href="#r33">[33]</a></sup> <strong>WABOT-1</strong> (Universidad de Waseda, 1972&ndash;73) fue el primer robot antropomórfico de tamaño real, capaz de caminar, sujetar objetos y conversar en japonés, a su manera. El <strong>perceptrón</strong> de Frank Rosenblatt (1958) fue la primera máquina que aprendía: una neurona de McCulloch&ndash;Pitts cuyos pesos se ajustaban automáticamente a partir de ejemplos etiquetados.<sup class="c"><a href="#r6">[6]</a></sup> En 1969, Minsky y Papert demostraron que un perceptrón de una sola capa no puede representar funciones tan simples como el o-exclusivo, y argumentaron que las versiones multicapa serían imposibles de entrenar.<sup class="c"><a href="#r7">[7]</a></sup> El argumento fue influyente, el financiamiento para las redes neuronales se desplomó y, junto con el informe Lighthill de 1973 en el Reino Unido, que concluyó que la IA no había cumplido sus promesas, produjo el <strong>primer invierno de la IA</strong>.

### 2.3 Sistemas expertos, medicina y el segundo invierno (1970&ndash;1993)

El campo se recuperó acotando su ambición. Los <strong>sistemas expertos</strong> codificaban el conocimiento de un especialista humano como cientos o miles de reglas SI&ndash;ENTONCES y razonaban con ellas. La medicina fue la vitrina. <strong>MYCIN</strong>, desarrollado en Stanford desde 1972, recomendaba terapia antibiótica para infecciones bacterianas a partir de unas 600 reglas, e introdujo los <em>factores de certeza</em> para razonar bajo incertidumbre &mdash; una regla podía concluir "el organismo es <em>Pseudomonas</em> con certeza 0.7".<sup class="c"><a href="#r8">[8]</a></sup> En una evaluación ciega, sus recomendaciones fueron calificadas como aceptables al menos con la misma frecuencia que las de los académicos especialistas en enfermedades infecciosas. Nunca se usó clínicamente &mdash; por razones legales, de flujo de trabajo y de computación &mdash;, pero estableció que una máquina podía igualar a los especialistas en un dominio acotado, y las preguntas que planteó (¿quién es responsable?, ¿cómo se explica el sistema a sí mismo?) son las preguntas que la FDA y la Unión Europea están regulando hoy. La Sección 3 retoma en código el estilo de razonamiento de MYCIN.

Los sistemas expertos se convirtieron en una industria multimillonaria en los años 1980 &mdash; la primera conferencia nacional de la American Association for Artificial Intelligence se realizó en Stanford en 1980, y se informó que <strong>XCON</strong> (R1), que configuraba pedidos de computadores VAX para Digital Equipment Corporation a partir de unas 2,500 reglas, le ahorraba a la empresa decenas de millones de dólares al año &mdash; y la industria colapsó a comienzos de los años 1990. La razón es instructiva: las reglas debían escribirse a mano, el <em>cuello de botella de la adquisición de conocimiento</em>; eran frágiles y fallaban abruptamente fuera de las situaciones que sus autores habían previsto; y no podían mejorar con la experiencia. Siguió el <strong>segundo invierno de la IA</strong>.

### 2.4 Aprender de los datos (1986&ndash;2011)

Dos desarrollos sentaron las bases del presente. En 1986, Rumelhart, Hinton y Williams popularizaron la <strong>retropropagación</strong> (backpropagation), el algoritmo que entrena redes neuronales multicapa propagando el error de salida hacia atrás a través de las capas &mdash; justamente lo que Minsky y Papert habían considerado inviable.<sup class="c"><a href="#r9">[9]</a></sup> En paralelo, estadísticos e informáticos construyeron la teoría <em>estadística</em> del aprendizaje a partir de datos: árboles de decisión (Unidad 4), máquinas de vectores de soporte y métodos bayesianos (Unidad 3), métodos de ensamble y la metodología rigurosa de los conjuntos de entrenamiento y de prueba (Suplemento S1). La victoria de Deep Blue de IBM sobre el campeón mundial de ajedrez en 1997 fue un hito de búsqueda y de ingeniería más que de aprendizaje, pero cambió la percepción pública. La década de 2000 llevó la IA a los hogares y las empresas de manera silenciosa: la aspiradora robot Roomba (2002) fue el primer robot autónomo de mercado masivo, y a mediados de la década los sistemas de recomendación de Netflix, Amazon y las redes sociales eran modelos de aprendizaje automático que atendían a cientos de millones de usuarios.

### 2.5 Aprendizaje profundo (2012&ndash;presente)

En 2012, una red neuronal convolucional, AlexNet, entrenada en unidades de procesamiento gráfico (GPU), redujo el error en el benchmark de reconocimiento de imágenes ImageNet de 26&nbsp;% a 15&nbsp;%, un margen al que ningún método anterior se había acercado.<sup class="c"><a href="#r10">[10]</a></sup> Los ingredientes eran antiguos &mdash; la arquitectura de la red data de 1989 &mdash; y el cambio fue cuantitativo: un millón de imágenes etiquetadas y un aumento de mil veces en capacidad de cómputo. Todo lo que vino después sigue la misma receta. Varios acontecimientos de la década pertenecen a la historia de la percepción pública más que a la de los métodos: Watson de IBM ganó el concurso de preguntas <em>Jeopardy!</em> en 2011 frente a los dos mejores campeones humanos, combinando cientos de componentes de lenguaje natural y de recuperación de información;<sup class="c"><a href="#r34">[34]</a></sup> en 2014 se declaró que un personaje de chatbot, "Eugene Goostman", había superado una prueba de Turing en la Royal Society al convencer al 33&nbsp;% de los jueces, en conversaciones de cinco minutos, de que era un niño ucraniano de 13 años &mdash; un resultado que la mayoría de los investigadores considera una muestra de la debilidad de la prueba más que de la fortaleza del programa &mdash;; y en 2018 Project Debater de IBM sostuvo un debate en vivo contra debatientes campeones. AlphaGo derrotó a un campeón mundial de Go en 2016 combinando redes profundas con búsqueda en árbol y aprendizaje por refuerzo;<sup class="c"><a href="#r11">[11]</a></sup> la arquitectura Transformer de 2017 hizo posible entrenar modelos de lenguaje con toda la web; AlphaFold2 resolvió en 2020 el problema de predicción de la estructura de proteínas, abierto durante cincuenta años, con una exactitud cercana a la experimental,<sup class="c"><a href="#r12">[12]</a></sup> un resultado que estudiaste en el curso de bioinformática; y los grandes modelos de lenguaje han alcanzado, desde 2022, el umbral de aprobación de los exámenes de habilitación médica.<sup class="c"><a href="#r4">[4]</a></sup>

### 2.6 IA generativa

Los sistemas que llevaron la IA a la atención pública después de 2022 pertenecen a una subfamilia, la <strong>IA generativa</strong>: modelos entrenados con colecciones muy grandes de contenido existente que producen contenido <em>nuevo</em> del mismo tipo. Hay tres modalidades de uso común.

**Texto a texto.** Los grandes modelos de lenguaje son redes neuronales entrenadas para predecir la palabra siguiente sobre corpus de billones de palabras; la arquitectura Transformer hizo que ese entrenamiento fuera paralelizable a gran escala.<sup class="c"><a href="#r35">[35]</a></sup> La clase menciona GPT (Generative Pre-trained Transformer, OpenAI) y BERT (Google). Ambos son objetos de distinto tipo y vale la pena mantener la distinción: los modelos GPT son generativos &mdash; producen texto &mdash;, mientras que BERT es un <em>codificador</em> entrenado para producir representaciones de texto para clasificación y búsqueda, y no genera texto.<sup class="c"><a href="#r36">[36]</a></sup> (Las diapositivas incluyen a BERT entre los modelos "generativos"; esta es la única corrección que esta unidad les hace &mdash; BERT es un codificador). Aplicaciones: asistentes, traducción, resumen, generación de código; en medicina, redacción de notas clínicas y los resultados de respuesta a preguntas citados más arriba.<sup class="c"><a href="#r4">[4]</a></sup>

**Texto a imagen, e imagen a video.** Los modelos de difusión producen imágenes fotorrealistas a partir de una instrucción de texto (prompt). La clase recuerda dos casos que obligaron a plantear la pregunta de la autoría: en 2022, una imagen generada con Midjourney ganó la categoría de arte digital de la Feria Estatal de Colorado, y en 2023 el fotógrafo Boris Eldagsen ganó un Sony World Photography Award con una imagen generada por IA y rechazó el premio, diciendo que la había presentado para comprobar si los concursos estaban preparados para ese tipo de obras. La generación de video (Emu Video de Meta, 2023; Sora de OpenAI, 2024) extiende los mismos modelos a imágenes en movimiento. El procesamiento de video en tiempo real &mdash; resíntesis de rostros, corrección de la mirada, reconocimiento de voz y traducción automática en una videollamada (NVIDIA Maxine es el ejemplo de la clase) &mdash; usa la misma familia de métodos.

**Por qué importa para este curso.** Los modelos generativos se construyen con los mismos componentes que los clasificadores que estudiarás &mdash; representación, función de pérdida, optimización, muestra de entrenamiento &mdash; y heredan los mismos modos de falla: reproducen los sesgos de sus datos de entrenamiento, fallan ante cambios en la distribución de los datos y se evalúan con benchmarks que pueden no reflejar el uso real. Agregan un modo de falla propio, la producción de contenido fluido, seguro de sí mismo y falso, y un problema social, tratado en la Sección 7.6: contenido que no se puede distinguir del auténtico.

<div class="box key"><h4>Lo que enseña la historia</h4>
<ul>
<li>Cada invierno siguió a un período en el que la capacidad se sobrestimó y la fragilidad se subestimó. Informar honestamente lo que un sistema <em>no puede</em> hacer es una competencia profesional, no pesimismo.</li>
<li>El cambio decisivo fue pasar de <strong>programar el conocimiento</strong> (sistemas expertos) a <strong>aprenderlo de los datos</strong>. Ese cambio explica por qué los datos &mdash; su cantidad, calidad, representatividad y gobernanza &mdash; son hoy la preocupación central de la ingeniería, y por qué las Secciones 4&ndash;7 tratan sobre datos.</li>
<li>El progreso ha sido impulsado tanto por el cómputo y los datos como por las ideas: la neurona de 1943, el perceptrón de 1958 y la retropropagación de 1986 están todos presentes en una red de 2024.</li>
</ul></div>

## 3. Los enfoques de la inteligencia artificial

La descripción del curso habla de "métodos de formalización del conocimiento, razonamiento con y sin incertidumbre, y técnicas de aprendizaje automático". Estas frases nombran los principales enfoques del campo, que coexisten en lugar de sucederse unos a otros.

### 3.1 IA simbólica (basada en conocimiento)

El enfoque simbólico representa el conocimiento de forma explícita, como enunciados lógicos o reglas, y obtiene conclusiones por inferencia. Sus herramientas son la lógica proposicional y de primer orden, las reglas de producción, las ontologías y los grafos de conocimiento. El razonamiento sin incertidumbre es la lógica clásica; el razonamiento <em>con</em> incertidumbre asigna grados de creencia a los enunciados, como hacían los factores de certeza de MYCIN o, en la formulación moderna, usa la teoría de probabilidades y las redes bayesianas (Unidad 3). Los sistemas simbólicos son transparentes &mdash; cada conclusión tiene una cadena de reglas trazable &mdash; y frágiles. En medicina sobreviven en las reglas de apoyo a la decisión clínica (alertas de interacciones entre fármacos, calculadoras de dosis) y en terminologías como SNOMED CT y la Gene Ontology, que conociste en bioinformática.

### 3.2 IA conexionista

El enfoque conexionista representa el conocimiento de forma implícita, distribuido en los pesos de una red de unidades simples, y lo adquiere mediante entrenamiento. Nada en una red entrenada corresponde a una regla legible. El enfoque es poderoso cuando el conocimiento es difícil de articular &mdash; reconocer un melanoma, transcribir voz &mdash; y opaco cuando se requiere una explicación. Las Unidades 6 y 8 están dedicadas a él.

### 3.3 IA estadística (probabilística) y aprendizaje automático

El enfoque estadístico trata el aprendizaje como estimación: dada una muestra de datos, ajustar un modelo que prediga bien sobre datos nuevos provenientes de la misma fuente, y cuantificar la incertidumbre de la predicción. Los árboles de decisión, los k vecinos más cercanos, las máquinas de vectores de soporte, Bayes ingenuo, la regresión y el agrupamiento (clustering) (Unidades 2&ndash;5) son métodos de aprendizaje estadístico. El enfoque conexionista es, en la práctica moderna, también estadístico &mdash; una red neuronal es un modelo estadístico muy flexible ajustado mediante optimización &mdash; y el término <em>aprendizaje automático</em> abarca a ambos.

### 3.4 IA evolutiva

El enfoque evolutivo trata la resolución de problemas como una búsqueda mediante selección natural simulada: se evalúa una población de soluciones candidatas, se seleccionan las más aptas, se recombinan y se mutan, y el ciclo se repite (Unidad 7). Se usa en problemas de optimización con paisajes objetivo accidentados &mdash; planificación, diseño y, en tu área, el diseño de moléculas con propiedades objetivo.

### 3.5 Reglas frente a reglas aprendidas, en código

El paso del enfoque simbólico al estadístico se puede mostrar en veinte líneas. El conjunto de datos es el que reaparece a lo largo de este curso: los datos de <strong>cáncer de mama de Wisconsin</strong> (Wisconsin breast-cancer). El paquete <code>mlbench</code><sup class="c"><a href="#r38">[38]</a></sup> incluye la versión citológica original de Wolberg &mdash; 699 punciones aspirativas con aguja fina (683 una vez eliminados los 16 registros con un valor faltante), cada una descrita por nueve atributos citológicos puntuados por un patólogo de 1 a 10 (grosor del cúmulo, uniformidad del tamaño celular, uniformidad de la forma celular, adhesión marginal, tamaño de la célula epitelial única, núcleos desnudos, cromatina suave, nucléolos normales y mitosis), y etiquetada como benigna o maligna.<sup class="c"><a href="#r13">[13]</a></sup>

<div class="box warn"><h4>Una nota sobre el conjunto de datos (diferencia con la edición en Python)</h4><p>La función <code>load_breast_cancer</code> de scikit-learn incluye el conjunto Wisconsin <em>diagnóstico</em>, posterior: 569 punciones aspirativas y 30 características morfométricas de valor real (radio, textura, perímetro&nbsp;&hellip;). El <code>mlbench::BreastCancer</code> de R es el conjunto Wisconsin <em>original</em>, anterior: más casos, pero nueve atributos citológicos puntuados con números enteros. La pedagogía es idéntica &mdash; la misma regla manual frente a la regla aprendida, el mismo clasificador del centroide más cercano &mdash;, pero los nombres de las características, los umbrales y las exactitudes exactas difieren de los del notebook de Python porque los dos conjuntos de datos son distintos. Cada número que aparece a continuación se calcula a partir del conjunto de datos de R que tienes frente a ti.</p></div>



In [ ]:
# Carga y depura los datos de cáncer de mama de Wisconsin (el conjunto de datos recurrente del curso).
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # elimina la columna Id
bc <- bc[complete.cases(bc), ]                             # elimina las 16 filas con un valor faltante
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # factor ordinal -> numérico 1..10
cat("rows x features:", nrow(bc), "x", ncol(bc) - 1, "\n")
table(bc$Class)



La tabla depurada tiene 683 punciones aspirativas completas descritas por nueve características; los recuentos por clase (444 benignas, 239 malignas) muestran un desbalance leve de aproximadamente 1&nbsp;:&nbsp;1.9. Antes de escribir cualquier regla, el bloque de código siguiente examina dos de las nueve características para ver si las clases se separan en absoluto.



In [ ]:
# ¿Ocupan las punciones malignas y benignas regiones distintas de dos características?
cols <- ifelse(bc$Class == "malignant", "#d2691e", "#2b5d9b")
plot(jitter(bc$Cell.size), jitter(bc$Bare.nuclei), col = cols, pch = 19, cex = .5,
     xlab = "uniformity of cell size (1-10)", ylab = "bare nuclei (1-10)",
     main = "Two cytological features (malignant = orange)")
legend("topleft", c("benign", "malignant"), col = c("#2b5d9b", "#d2691e"), pch = 19, bty = "n", cex = .9)



<p class="caption"><b>Figura 2.</b> Dos de los nueve puntajes citológicos, con una pequeña perturbación aleatoria (jitter) para que los valores enteros superpuestos sean visibles. Las punciones benignas (azul) se agrupan en valores bajos de tamaño celular y pocos núcleos desnudos; las malignas (naranja) se extienden hacia valores altos de ambos. Las clases ocupan regiones distintas &mdash; la separabilidad que hace posible una regla.</p>

El primer bloque de código escribe una regla de clasificación como lo haría un ingeniero del conocimiento de los años 1980 &mdash; preguntándole a un patólogo qué características importan y eligiendo los umbrales a mano. El segundo deja que un algoritmo encuentre la regla a partir de los datos.



In [ ]:
# Una regla escrita a mano, al estilo de un sistema experto: los núcleos grandes e irregulares son sospechosos.
is_malignant <- bc$Class == "malignant"
rule_says_malignant <- (bc$Cell.size > 3) & (bc$Bare.nuclei > 3)   # umbrales elegidos a mano
accuracy <- mean(rule_says_malignant == is_malignant)
cat(sprintf("Cases: %d  (malignant %d, benign %d)\n", nrow(bc), sum(is_malignant), sum(!is_malignant)))
cat(sprintf("Hand-written rule agrees with the biopsy in %.1f%% of cases\n", 100 * accuracy))



La regla manual &mdash; "tamaño celular mayor que 3 Y núcleos desnudos mayor que 3 significa maligno" &mdash; concuerda con la biopsia en el 89.0&nbsp;% de los 683 casos. Eso ya es útil, y se escribió sin ningún ajuste. El bloque de código siguiente deja que los datos elijan las características y los umbrales.



In [ ]:
# La misma tarea, pero la regla se APRENDE de los datos (un árbol de decisión de profundidad 2; la Unidad 4 abre esta caja).
lab <- factor(ifelse(is_malignant, "malignant", "benign"))
tree <- rpart(lab ~ ., data = data.frame(bc[, 1:9], lab = lab), method = "class",
              control = rpart.control(maxdepth = 2, cp = 0, xval = 0))   # rpart = el CART de R
print(tree)
cat(sprintf("\nLearned rule agrees with the biopsy in %.1f%% of cases (on the data it was fitted to)\n",
            100 * mean(predict(tree, type = "class") == lab)))



El árbol aprendido<sup class="c"><a href="#r40">[40]</a></sup> divide primero según el tamaño celular, luego según los núcleos desnudos y la forma celular, y concuerda con la biopsia en el 95.5&nbsp;% de los casos. El bloque de código de abajo lo dibuja.



In [ ]:
rpart.plot(tree, type = 2, extra = 2, box.palette = list("#dfe7f1", "#fbe3d3"),
           main = "A depth-2 decision tree learned from the data")



<p class="caption"><b>Figura 3.</b> La regla aprendida dibujada como árbol. Cada nodo interno es una prueba SI&ndash;ENTONCES sobre una característica; cada hoja entrega la clase predicha y el recuento de casos que abarca. La regla tiene la misma forma SI&ndash;ENTONCES legible que la escrita a mano, pero sus características y umbrales fueron elegidos por un algoritmo que examinó los 683 casos y las nueve características.</p>

La regla aprendida tiene la misma forma SI&ndash;ENTONCES que la escrita a mano &mdash; sigue siendo simbólica y legible &mdash;, pero sus características y umbrales fueron elegidos por un algoritmo, y concuerda con la biopsia con mayor frecuencia. Dos advertencias introducen los dos temas metodológicos del curso. Primero, la exactitud de la regla aprendida se midió sobre los mismos casos con los que se ajustó, lo que la favorece: el Suplemento S1 explica por qué y cómo medir honestamente. Segundo, el <em>costo</em> de los dos tipos de error no es el mismo &mdash; un tumor maligno clasificado como benigno es un cáncer no detectado; uno benigno clasificado como maligno es una biopsia innecesaria &mdash; y la "exactitud" los trata como iguales. Ten presentes ambas advertencias; reaparecen en todas las unidades.

<div class="box key"><h4>Ejercicio 1</h4><p>Cambia los umbrales de la regla escrita a mano (prueba tamaño celular 2 y núcleos desnudos 2, luego tamaño celular 4 y 5) y observa cómo cambia la concordancia. Luego cambia el <code>maxdepth</code> del árbol a 1 y a 4. ¿Por qué un árbol más profundo concuerda más con la biopsia en estos datos, y por qué eso debería hacerte sospechar en lugar de alegrarte?</p></div>

## 4. Reconocimiento de patrones

### 4.1 Definición

El <strong>reconocimiento de patrones</strong> es la disciplina científica cuyo objetivo es la clasificación de objetos en categorías o clases, donde los objetos pueden ser imágenes, señales, mediciones o cualquier otro tipo de observación.<sup class="c"><a href="#r14">[14]</a></sup> La palabra <em>patrón</em> designa una regularidad en los datos que es informativa sobre una categoría: la combinación particular de tamaño, forma y textura nuclear que distingue una célula maligna de una benigna es un patrón; la secuencia de deflexiones de voltaje que distingue la fibrilación auricular del ritmo sinusal en un electrocardiograma es un patrón. La disciplina es anterior al término "aprendizaje automático" &mdash; sus libros de texto datan de los años 1970 &mdash; y hoy ambos se superponen casi por completo: el aprendizaje automático aporta los métodos y el reconocimiento de patrones aporta la formulación del problema.

### 4.2 El pipeline

Todo sistema de reconocimiento de patrones, desde un lector óptico de caracteres de los años 1970 hasta un modelo de radiología de 2024, tiene las mismas etapas:

1. **Sensado** &mdash; una medición física produce datos crudos: una cámara produce intensidades de píxeles, un electrodo produce voltajes, un espectrómetro de masas produce intensidades por razón masa/carga, un cuestionario produce respuestas.
2. **Preprocesamiento y segmentación** &mdash; se elimina el ruido y se aísla el objeto de interés: el núcleo se segmenta del fondo, el latido cardíaco se recorta del registro continuo.
3. **Extracción de características** &mdash; el objeto se describe mediante un vector de números de longitud fija, el <strong>vector de características</strong>: radio, textura, perímetro&nbsp;&hellip; para el núcleo. Este es el paso que convierte una cosa en matemática, y aquel en el que entra el conocimiento del dominio. En el aprendizaje profundo las características se aprenden en lugar de diseñarse, pero el vector sigue ahí &mdash; es la activación de la penúltima capa de la red.
4. **Clasificación** &mdash; una función asigna el vector de características a una clase (o a una probabilidad sobre las clases). La función es el <em>clasificador</em>; sus parámetros se fijan aprendiendo a partir de la muestra de entrenamiento.
5. **Posprocesamiento y decisión** &mdash; la clase de salida se combina con costos, contexto y otra evidencia para producir una acción: biopsia o seguimiento, alarma o ausencia de alarma.

Este curso se concentra en las etapas 3 y 4 y en cómo evaluar el conjunto. La etapa 1 es el dominio de los cursos de instrumentación de tu carrera, y la etapa 5 es el dominio de la práctica clínica y de las políticas de salud; un bioingeniero se sitúa en la interfaz y debe comprender las cinco.

### 4.3 Representación: el vector de características y la matriz de datos

Un objeto descrito por \( p \) características medidas es un punto en un espacio de dimensión \( p \), escrito como un vector \( \mathbf{x} = (x_1, x_2, \dots, x_p) \). Una muestra de \( n \) objetos es una matriz \( \mathbf{X} \) de \( n \) filas (objetos, observaciones, instancias, ejemplos, casos &mdash; los términos son sinónimos) y \( p \) columnas (características, variables, atributos, descriptores &mdash; también sinónimos):

\[ \mathbf{X} = \begin{pmatrix} x_{11} & x_{12} & \cdots & x_{1p} \\ x_{21} & x_{22} & \cdots & x_{2p} \\ \vdots & & \ddots & \vdots \\ x_{n1} & x_{n2} & \cdots & x_{np} \end{pmatrix}, \qquad \mathbf{y} = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix} \]

Cuando cada objeto tiene además una clase conocida \( y_i \), el par \( (\mathbf{X}, \mathbf{y}) \) es un conjunto de datos <strong>etiquetado</strong> y el vector de clases \( \mathbf{y} \) es el <em>objetivo</em> (target), la <em>etiqueta</em> o la <em>respuesta</em>. La convención "las filas son observaciones, las columnas son características" es universal en el <code>data.frame</code> de R, en scikit-learn, en pandas y en la literatura estadística; un conjunto de datos en el que está invertida debe transponerse antes de aplicar cualquier método. En los datos de cáncer de mama, \( n = 683 \) y \( p = 9 \):



In [ ]:
# La matriz de datos X (683 x 9) y el vector de etiquetas y; las primeras filas de seis características.
X <- bc[, 1:9]; y <- bc$Class
cat("dim(X) (n objects, p features):", dim(X)[1], "x", dim(X)[2], "| length(y):", length(y), "\n")
cat("class labels and counts:\n"); print(table(y))
head(X[, 1:6], 5)



Cada fila es una punción aspirativa; cada columna es un puntaje citológico de 1 a 10. El primer paciente tiene un grosor del cúmulo de 5 y una uniformidad del tamaño celular de 1 &mdash; puntajes bajos, coherentes con la etiqueta benigna. El conjunto de datos tiene 239 casos malignos y 444 benignos, el desbalance leve señalado más arriba; en problemas de tamizaje donde la prevalencia de la enfermedad es de 1&nbsp;%, ese desbalance se convierte en la dificultad dominante (Suplemento S1).

<div class="box bridge"><h4>Puente con la bioinformática</h4><p>Ya has construido matrices de datos sin llamarlas así. Un alineamiento múltiple de secuencias es una matriz cuyas filas son secuencias y cuyas columnas son posiciones; una matriz BLOSUM es una característica de pares de residuos; los descriptores moleculares de la Unidad 4 de BGM324 (peso molecular, cLogP, TPSA, enlaces rotables&nbsp;&hellip;) son un vector de características de una molécula, y un modelo QSAR es un sistema de reconocimiento de patrones cuyos objetos son moléculas y cuyo objetivo es la actividad. Los métodos de este curso se aplican a esas matrices sin cambios.</p></div>

### 4.4 Tipos de datos

El tipo de una característica determina qué operaciones matemáticas tienen sentido y, por lo tanto, qué métodos se pueden aplicar. Confundir los tipos es el error más común en el trabajo aplicado.

| Tipo | Definición | Operaciones admisibles | Ejemplos clínicos |
|---|---|---|---|
| **Continuo** (de intervalo o de razón) | Números reales; las diferencias y (en escalas de razón) los cocientes tienen sentido | media, desviación estándar, distancia, correlación | glucosa (mg/dL), radio nuclear (µm), presión sistólica, edad |
| **Discreto** (conteo) | Enteros no negativos provenientes de un conteo | las mismas que los continuos, con cuidado en conteos pequeños | número de hospitalizaciones previas, ganglios linfáticos examinados, repeticiones CAG |
| **Binario** (dicotómico) | Dos estados | proporción, concordancia | sexo al nacer, fumador/no fumador, evento/sin evento |
| **Nominal** (no ordenado) | Etiquetas sin orden | moda, frecuencia; **sin media, sin distancia** | grupo sanguíneo, comuna de residencia, código CIE-10, nucleótido (A/C/G/T) |
| **Ordinal** (ordenado) | Etiquetas con un orden pero sin espaciado fijo | mediana, estadísticos de rangos; en principio **sin media** | estadio tumoral I&ndash;IV, dolor 0&ndash;10, Escala de Coma de Glasgow |
| **Texto, imagen, señal, secuencia** | Datos crudos estructurados | deben convertirse mediante extracción de características | notas clínicas, radiografías, ECG, ADN |

Los enunciados negativos importantes están en negrita: no existe un grupo sanguíneo medio, y la distancia entre los grupos sanguíneos A y B no está definida. Sin embargo, casi todos los métodos de este curso calculan distancias o medias. Por lo tanto, las características categóricas deben <em>codificarse</em> como números de una forma que no invente un orden: una característica nominal con \( k \) niveles se convierte en \( k \) columnas indicadoras binarias (<strong>codificación one-hot</strong>); una característica ordinal puede asignarse a sus rangos si el analista acepta que el espaciado se trate como uniforme. El bloque de código siguiente construye una pequeña tabla clínica que contiene todos los tipos y muestra las codificaciones. (Los nueve puntajes de cáncer de mama son ellos mismos ordinales; por eso este curso los trata como números solo después de señalar que se está haciendo el supuesto de espaciado uniforme).



In [ ]:
# Una tabla clínica sintética que contiene todos los tipos de datos de la Sección 4.4.
set.seed(0); n <- 8
clinic <- data.frame(
  age_years        = sample(35:79, n, replace = TRUE),                               # continuo (escala de razón)
  glucose_mg_dL    = round(rnorm(n, 105, 20)),                                        # continuo
  prior_admissions = rpois(n, 1.2),                                                   # conteo discreto
  smoker           = sample(c(TRUE, FALSE), n, replace = TRUE),                       # binario
  blood_group      = sample(c("O", "A", "B", "AB"), n, replace = TRUE),              # nominal
  tumour_stage     = factor(sample(c("I","II","III","IV"), n, replace = TRUE),
                            levels = c("I","II","III","IV"), ordered = TRUE),          # ordinal
  stringsAsFactors = FALSE)
str(clinic)
clinic


In [ ]:
# Codificaciones que respetan el tipo: one-hot para la característica nominal, códigos de rango para la ordinal.
onehot <- model.matrix(~ blood_group - 1, clinic)                 # 4 columnas indicadoras, sin orden
colnames(onehot) <- sub("blood_group", "bg_", colnames(onehot))
encoded <- data.frame(clinic[c("age_years","glucose_mg_dL","prior_admissions")],
                      smoker = as.integer(clinic$smoker),
                      onehot,
                      stage_code = as.integer(clinic$tumour_stage))  # ordinal: 1,2,3,4 preserva el orden
cat("From", ncol(clinic), "clinical variables to", ncol(encoded), "numerical features\n")
head(encoded)



Observa lo que ocurrió con <code>blood_group</code>: una columna con cuatro etiquetas se convirtió en cuatro columnas de ceros y unos, de modo que la distancia euclidiana entre dos grupos sanguíneos distintos cualesquiera es la misma (\( \sqrt{2} \)) y ningún grupo queda "entre" otros dos. Si hubiéramos codificado O&nbsp;=&nbsp;1, A&nbsp;=&nbsp;2, B&nbsp;=&nbsp;3, AB&nbsp;=&nbsp;4, todo método basado en distancias habría tratado a AB como tres pasos más lejos de O que A, una afirmación sin ningún significado biológico. La característica ordinal <code>tumour_stage</code>, en cambio, se codifica de manera defendible como 1&ndash;4, porque el orden es real; el analista acepta que los pasos se traten como iguales. Convertir texto libre en características &mdash; contando palabras, mediante diccionarios de términos clínicos o con los embeddings de un modelo de lenguaje &mdash; es una disciplina en sí misma (el procesamiento de lenguaje natural) que la Unidad 9 aborda brevemente.

### 4.5 Las imágenes y las señales también son vectores de características

Una imagen es una grilla de intensidades de píxeles; aplanar la grilla fila por fila entrega un vector cuya longitud es el número de píxeles. Un electrocardiograma de un segundo muestreado a 500&nbsp;Hz ya es un vector de 500 números, o puede resumirse mediante un puñado de características diseñadas (frecuencia cardíaca, duración del QRS, potencia espectral en una banda). Los dos bloques de código siguientes ilustran este punto. Como R no incluye un conjunto de datos de dígitos manuscritos, <em>simulamos</em> pequeñas imágenes de 8&nbsp;&times;&nbsp;8 &mdash; un sustituto del benchmark clásico de dígitos<sup class="c"><a href="#r15">[15]</a></sup> lo bastante pequeño para inspeccionarlo a simple vista &mdash; y una señal fisiológica sintética. Cada clase simulada es un trazo plantilla más ruido.



In [ ]:
# Simula imágenes de "trazos" de 8x8 de cuatro clases; una imagen es un punto en un espacio de 64 dimensiones.
set.seed(0)
templates <- list(
  vbar  = {m <- matrix(0,8,8); m[,4:5] <- 13; m},            # trazo vertical
  hbar  = {m <- matrix(0,8,8); m[4:5,] <- 13; m},            # trazo horizontal
  cross = {m <- matrix(0,8,8); m[,4:5] <- 13; m[4:5,] <- 13; m},  # signo más
  diag  = {m <- matrix(0,8,8); m[cbind(1:8,1:8)] <- 13; m[cbind(1:8,pmin(1:8+1,8))] <- 13; m})
classes <- names(templates)
make_sample <- function(tpl) pmax(0, pmin(16, round(tpl + rnorm(64, 0, 2))))
Ximg <- do.call(rbind, lapply(rep(classes, each = 40), function(k) as.vector(make_sample(templates[[k]]))))
yimg <- factor(rep(classes, each = 40), levels = classes)
cat("Image shape: 8 x 8 -> vector length:", ncol(Ximg), "| data matrix X:", nrow(Ximg), "x", ncol(Ximg), "\n")

show_img <- function(v, main="") image(t(matrix(v,8,8)[8:1,]), col = gray(seq(1,0,length=17)),
                                       zlim = c(0,16), axes = FALSE, main = main)
layout(matrix(c(1,2,2,2), 1)); par(mar = c(3,2,2,1))
show_img(Ximg[1,], main = "class 'vbar' as an 8 x 8 image")
barplot(Ximg[1,], col = "#2b5d9b", border = NA, space = 0,
        main = "the same object as a 64-element feature vector (pixels, row by row)",
        xlab = "vector index", ylab = "intensity (0-16)")


In [ ]:
# La 'imagen promedio' de cada clase: el patrón de la clase (plantilla) en el espacio de píxeles.
par(mfrow = c(1, length(classes)), mar = c(1,1,2,1))
for (k in classes) show_img(colMeans(Ximg[yimg == k, ]), main = k)



<p class="caption"><b>Figura 4.</b> Arriba: una imagen simulada "vbar" mostrada como una figura de 8&nbsp;&times;&nbsp;8 y como el vector de 64 elementos que efectivamente ve un clasificador. Abajo: la media de las 40 imágenes de cada clase. Promediar sobre una clase produce una plantilla limpia del patrón &mdash; la regularidad que comparten todos los miembros de una clase y que los distingue de los demás. El clasificador más simple de la Sección 5 hace exactamente esto: compara un objeto nuevo con la plantilla de cada clase.</p>



In [ ]:
# Una señal como patrón: 'latidos' sintéticos de dos ritmos, y un resumen de dos características de cada uno.
set.seed(1); fs <- 250; seconds <- 4; t <- seq(0, seconds, by = 1/fs)
beat_train <- function(rate_bpm, jitter, n_seg) t(sapply(1:n_seg, function(i) {
  peaks <- cumsum(rnorm(20, 60/rate_bpm, jitter)); peaks <- peaks[peaks < seconds]
  rowSums(sapply(peaks, function(pk) exp(-((t - pk)/0.02)^2))) + rnorm(length(t), 0, 0.05) }))
regular <- beat_train(70, 0.01, 30); irregular <- beat_train(95, 0.15, 30)
find_peaks <- function(x, h, d) { c <- which(x[2:(length(x)-1)] > x[1:(length(x)-2)] &
  x[2:(length(x)-1)] >= x[3:length(x)] & x[2:(length(x)-1)] > h) + 1
  keep <- integer(0); last <- -Inf; for (i in c) if (i - last >= d) { keep <- c(keep, i); last <- i }; keep }
features <- function(sig) { idx <- find_peaks(sig, 0.5, round(0.25*fs)); rr <- diff(t[idx]); c(mean(rr), sd(rr)) }
F <- t(apply(rbind(regular, irregular), 1, features)); labs <- rep(0:1, each = 30)
par(mfrow = c(1, 2), mar = c(4,4,2,1))
plot(t, regular[1,] + 1.6, type = "l", col = "#2b5d9b", lwd = .8, xlab = "time (s)", ylab = "",
     ylim = range(c(regular[1,] + 1.6, irregular[1,])), yaxt = "n", main = "raw signal: 1000 samples / segment")
lines(t, irregular[1,], col = "#d2691e", lwd = .8)
legend("topright", c("regular (70 bpm)","irregular (95 bpm)"), col = c("#2b5d9b","#d2691e"), lwd = 1, bty = "n", cex = .8)
plot(F[labs==0,1], F[labs==0,2], col = "#2b5d9b", pch = 19, cex = .7, xlab = "mean beat interval (s)",
     ylab = "interval variability (s)", xlim = range(F[,1]), ylim = range(F[,2]),
     main = "feature space: 2 numbers / segment")
points(F[labs==1,1], F[labs==1,2], col = "#d2691e", pch = 19, cex = .7)



<p class="caption"><b>Figura 5.</b> Izquierda: un segmento de cada ritmo sintético, de 1,000 muestras de longitud. Derecha: los mismos 60 segmentos después de la extracción de características &mdash; cada segmento es ahora un punto descrito por dos números, el intervalo medio entre latidos y su variabilidad &mdash; y los dos ritmos ocupan regiones separadas. La extracción de características redujo la dimensión de 1,000 a 2 conservando la información que importa para la tarea. Si conserva lo suficiente es una pregunta empírica que responde el desempeño del clasificador, nunca la sola inspección visual.</p>

<div class="box key"><h4>Idea clave</h4><p>Un patrón es una <em>región del espacio de características</em> ocupada preferentemente por una clase. Un clasificador es una regla que divide el espacio de características en regiones y asigna una clase a cada una. El arte del reconocimiento de patrones consiste en encontrar una representación en la que las clases ocupen regiones separables, y una regla que encuentre las fronteras a partir de una muestra finita sin ajustarse a los accidentes de la muestra.</p></div>

### 4.6 La muestra de entrenamiento

La <strong>muestra de entrenamiento</strong> (conjunto de entrenamiento) es el conjunto de datos etiquetado a partir del cual se aprenden los parámetros del clasificador. Tres de sus propiedades determinan todo lo que sigue en el curso.

**Es una muestra, no la población.** Las punciones aspirativas de Wisconsin se recolectaron en un solo hospital, se obtuvieron sus imágenes con un único protocolo y fueron etiquetadas por un solo servicio de anatomía patológica. Al clasificador se le pedirá clasificar punciones de otros hospitales, microscopios y poblaciones. El aprendizaje es <em>inductivo</em>: generaliza de los casos vistos a los casos no vistos, y esa generalización solo se justifica si los casos no vistos provienen de la misma distribución que los vistos. Cuando no es así &mdash; un escáner distinto, una población de mayor edad, una prevalencia distinta de la enfermedad &mdash;, la situación se denomina <strong>cambio en la distribución de los datos (dataset shift)</strong>, y el desempeño se degrada, a menudo de forma silenciosa. La Sección 7.3 muestra un mecanismo.

**Sus etiquetas definen la tarea.** El clasificador aprende a reproducir lo que sea que diga la etiqueta, no lo que el analista pretende que signifique. Si la etiqueta es "resultado de la biopsia", el sistema aprende a predecir biopsias; si la etiqueta es "costo de atención de salud", el sistema aprende a predecir el costo, aunque el analista lo llame "necesidad de salud". La Sección 7.4 muestra cuánto pueden divergir ambas cosas.

**Su tamaño limita lo que se puede aprender.** Con \( n \) ejemplos y \( p \) características, un modelo flexible puede memorizar la muestra sin aprender el patrón. El cociente \( n/p \), la complejidad del modelo y el ruido en las etiquetas determinan en conjunto cuánto de lo que aprende el modelo es patrón y cuánto es accidente; la metodología del Suplemento S1 existe para medir la diferencia. La Sección 5.3 muestra la dependencia de \( n \) con el clasificador más simple posible.

## 5. El clasificador más simple, a mano y en código

Para concretar el pipeline antes de que el curso introduzca algoritmos reales, esta sección construye el clasificador más simple que existe: el clasificador del <strong>centroide más cercano</strong> (de distancia mínima). Su entrenamiento consiste en calcular el vector de características medio &mdash; el <em>centroide</em> &mdash; de cada clase. Su regla de decisión asigna un objeto nuevo a la clase cuyo centroide está más cerca en distancia euclidiana:

<div class="eq">\[ \hat{y}(\mathbf{x}) = \arg\min_{c} \; \lVert \mathbf{x} - \boldsymbol{\mu}_c \rVert_2, \qquad \boldsymbol{\mu}_c = \frac{1}{n_c} \sum_{i:\, y_i = c} \mathbf{x}_i \]
<span class="where">donde \( \boldsymbol{\mu}_c \) es el centroide de la clase \( c \), calculado a partir de los \( n_c \) objetos de entrenamiento de esa clase, y \( \lVert\cdot\rVert_2 \) es la norma euclidiana. La frontera de decisión entre dos clases es la mediatriz del segmento que une sus centroides &mdash; una recta en dos dimensiones, un plano en tres y un hiperplano en general.</span></div>

### 5.1 Ejemplo resuelto a mano

Toma dos características ilustrativas de una punción aspirativa, \( x_1 \) y \( x_2 \) (en una escala arbitraria común), y seis casos de entrenamiento, tres de cada clase, con valores redondeados para el cálculo a mano:

| caso | \( x_1 \) | \( x_2 \) | clase |
|---|---|---|---|
| 1 | 12 | 2 | benigno |
| 2 | 13 | 3 | benigno |
| 3 | 11 | 1 | benigno |
| 4 | 19 | 9 | maligno |
| 5 | 21 | 11 | maligno |
| 6 | 17 | 7 | maligno |

**Entrenamiento.** Los centroides son las medias por clase: \( \boldsymbol{\mu}_{B} = \left(\tfrac{12+13+11}{3}, \tfrac{2+3+1}{3}\right) = (12, 2) \) y \( \boldsymbol{\mu}_{M} = \left(\tfrac{19+21+17}{3}, \tfrac{9+11+7}{3}\right) = (19, 9) \).

**Clasificación** de un caso nuevo \( \mathbf{x} = (15, 4) \): \( d_B = \sqrt{(15-12)^2 + (4-2)^2} = \sqrt{9 + 4} = \sqrt{13} \approx 3.61 \) y \( d_M = \sqrt{(15-19)^2 + (4-9)^2} = \sqrt{16 + 25} = \sqrt{41} \approx 6.40 \). Como \( d_B < d_M \), el caso se clasifica como <strong>benigno</strong>. El bloque de código de abajo confirma la aritmética y dibuja la frontera de decisión.



In [ ]:
# Clasificador del centroide más cercano sobre los seis casos resueltos a mano, y su frontera de decisión.
train <- rbind(c(12,2), c(13,3), c(11,1), c(19,9), c(21,11), c(17,7)); cls <- c("B","B","B","M","M","M")
mu_B <- colMeans(train[cls=="B",]); mu_M <- colMeans(train[cls=="M",])
new <- c(15, 4); d_B <- sqrt(sum((new-mu_B)^2)); d_M <- sqrt(sum((new-mu_M)^2))
cat(sprintf("centroid benign = (%g, %g), centroid malignant = (%g, %g)\n", mu_B[1],mu_B[2],mu_M[1],mu_M[2]))
cat(sprintf("distance to benign = %.2f, to malignant = %.2f  ->  predicted: %s\n",
            d_B, d_M, ifelse(d_B < d_M, "benign", "malignant")))
plot(train, col = ifelse(cls=="B","#2b5d9b","#d2691e"), pch = 19, cex = 1.4, xlim = c(9,23), ylim = c(-1,13),
     xlab = "feature x1", ylab = "feature x2", main = "Nearest-centroid classifier on six training cases")
points(mu_B[1], mu_B[2], pch = 4, lwd = 3, cex = 2, col = "#2b5d9b")
points(mu_M[1], mu_M[2], pch = 4, lwd = 3, cex = 2, col = "#d2691e")
points(new[1], new[2], pch = 8, cex = 2, lwd = 2, col = "#3a7d44")
mid <- (mu_B + mu_M)/2; dir <- mu_M - mu_B; perp <- c(-dir[2], dir[1])
pts <- rbind(mid - 1.2*perp, mid + 1.2*perp); lines(pts, lty = 2)
legend("topleft", c("benign","malignant","centroids (x)","new case (8)","boundary"),
       col = c("#2b5d9b","#d2691e","black","#3a7d44","black"),
       pch = c(19,19,4,8,NA), lty = c(NA,NA,NA,NA,2), bg = "white", box.col = NA, cex = .8)



<p class="caption"><b>Figura 6.</b> Los seis casos de entrenamiento, los dos centroides (cruces), el caso nuevo (estrella) y la frontera de decisión, que es la mediatriz del segmento que une los centroides. Todo lo que queda en el lado inferior izquierdo se clasifica como benigno, y todo lo que queda en el lado superior derecho, como maligno. Entrenar un clasificador del centroide más cercano significa calcular dos medias; la frontera se deriva de ellas.</p>

### 5.2 El mismo clasificador sobre el conjunto de datos completo

El ejemplo a mano usó seis casos elegidos por ser limpios. En la muestra real de 683, las clases se superponen &mdash; algunos tumores malignos tienen puntajes bajos, algunos benignos los tienen altos &mdash; y ninguna recta las separa perfectamente. El bloque de código entrena el clasificador del centroide más cercano con un 70&nbsp;% de los casos elegido al azar y lo prueba con el 30&nbsp;% restante, que nunca vio. La razón para reservar casos es el tema del Suplemento S1; aquí el objetivo es ver funcionar un clasificador real y leer correctamente su resultado.



In [ ]:
# Centroide más cercano sobre dos características reales, entrenado con el 70% de los casos y probado con el otro 30%.
set.seed(0)
feat2 <- c("Cl.thickness", "Cell.size")
X2 <- as.matrix(bc[, feat2]); yc <- bc$Class
idx <- sample(nrow(bc), round(0.7 * nrow(bc)))
cent <- t(sapply(levels(yc), function(c) colMeans(X2[idx, ][yc[idx] == c, ])))
euclid <- function(M, v) sqrt(rowSums(sweep(M, 2, v)^2))
predict_nc <- function(M) levels(yc)[apply(cbind(euclid(M, cent["benign",]), euclid(M, cent["malignant",])), 1, which.min)]
pred <- predict_nc(X2[-idx, ])
cat("centroids (rows = classes):\n"); print(round(cent, 2))
cat(sprintf("accuracy on the %d held-out cases: %.3f\n", nrow(bc) - length(idx),
            mean(pred == as.character(yc[-idx]))))
# regiones de decisión sobre el plano
gx <- seq(0.5, 10.5, length = 220); gy <- seq(0.5, 10.5, length = 220)
grid <- expand.grid(Cl.thickness = gx, Cell.size = gy)
Z <- matrix(predict_nc(as.matrix(grid)) == "malignant", length(gx))
image(gx, gy, Z, col = c("#dfe7f1", "#fbe3d3"), xlab = "clump thickness (1-10)", ylab = "cell size (1-10)",
      main = "Nearest-centroid decision regions (205 test cases)")
te <- X2[-idx, ]; tc <- yc[-idx]
points(jitter(te[tc=="benign",1]), jitter(te[tc=="benign",2]), col = "#2b5d9b", pch = 19, cex = .5)
points(jitter(te[tc=="malignant",1]), jitter(te[tc=="malignant",2]), col = "#d2691e", pch = 19, cex = .5)
points(cent[,1], cent[,2], pch = 4, lwd = 3, cex = 2, col = c("#2b5d9b","#d2691e"))



<p class="caption"><b>Figura 7.</b> Las dos regiones de decisión y los 205 casos de prueba (con jitter). La mayoría de los casos benignos (azul) cae en la región azul y la mayoría de los malignos (naranja) en la región naranja, pero una franja de casos cerca de la frontera se clasifica mal en ambas direcciones. La exactitud impresa sobre la figura &mdash; alrededor de 0.93 &mdash; es la fracción de casos de prueba que quedan en el lado correcto. Dos características y dos medias ya recuperan más de nueve de cada diez resultados de biopsia; el resto del curso trata sobre los casos restantes, y sobre saber <em>cuáles</em> son.</p>



In [ ]:
# Ejercicio 2 en código: ¿ayuda a este clasificador usar las nueve características?
cent9 <- t(sapply(levels(yc), function(c) colMeans(as.matrix(X[idx, ])[yc[idx] == c, ])))
d9 <- sapply(levels(yc), function(c) euclid(as.matrix(X[-idx, ]), cent9[c, ]))
pred9 <- levels(yc)[apply(d9, 1, which.min)]
cat(sprintf("test accuracy with 2 features: 0.927   with all 9 features: %.3f\n",
            mean(pred9 == as.character(yc[-idx]))))



Usar los nueve puntajes citológicos eleva la exactitud sobre los casos reservados de 0.927 a 0.951. 

<div class="box key"><h4>Ejercicio 2</h4><p>Reemplaza las dos características por <code>"Marg.adhesion"</code> y <code>"Mitoses"</code> y vuelve a ejecutar el clasificador de la Sección 5.2; la exactitud baja, porque esas dos características separan peor las clases. Observa una diferencia con la edición en Python: allí, agregar las 30 características diagnósticas <em>perjudicaba</em> al clasificador del centroide más cercano, porque características como "mean area" (que supera los 2,500) dominaban la distancia euclidiana. Aquí los nueve puntajes comparten la escala de 1&ndash;10, de modo que ninguna característica domina y más características ayudan. El escalamiento de características (Suplemento S0 y Unidad 3) es lo que hace que las dos ediciones se comporten de manera distinta.</p></div>

### 5.3 Por qué importa el tamaño de la muestra de entrenamiento

La Sección 4.6 afirmó que el tamaño de la muestra de entrenamiento limita lo que se puede aprender. El bloque de código de abajo lo mide: el clasificador del centroide más cercano se entrena con muestras de tamaño creciente extraídas al azar de los 478 casos de entrenamiento, se prueba con los mismos 205 casos reservados, y el experimento se repite 50 veces para cada tamaño a fin de promediar la suerte del sorteo.



In [ ]:
# Curva de aprendizaje: exactitud de prueba en función del número de casos de entrenamiento (50 sorteos por tamaño).
set.seed(0)
sizes <- c(4, 8, 16, 32, 64, 128, 256, length(idx)); ytr <- droplevels(yc[idx]); Xtr <- X2[idx, ]; Xte <- X2[-idx, ]; yte <- as.character(yc[-idx])
acc <- matrix(0, length(sizes), 50)
for (i in seq_along(sizes)) for (r in 1:50) {
  repeat { s <- sample(length(ytr), sizes[i]); if (length(unique(ytr[s])) == 2) break }
  ce <- t(sapply(levels(ytr), function(c) colMeans(Xtr[s, ][ytr[s] == c, , drop = FALSE])))
  pr <- levels(ytr)[apply(cbind(euclid(Xte, ce["benign",]), euclid(Xte, ce["malignant",])), 1, which.min)]
  acc[i, r] <- mean(pr == yte) }
summary_lc <- data.frame(n_train = sizes, mean_accuracy = round(rowMeans(acc), 3),
                         worst_draw = round(apply(acc, 1, min), 3), best_draw = round(apply(acc, 1, max), 3))
plot(sizes, rowMeans(acc), type = "n", log = "x", ylim = c(0.5, 1), xlab = "number of training cases",
     ylab = "accuracy on 205 test cases", main = "Learning curve of the nearest-centroid classifier")
polygon(c(sizes, rev(sizes)), c(apply(acc,1,quantile,.05), rev(apply(acc,1,quantile,.95))),
        col = "#dfe7f1", border = NA)
lines(sizes, rowMeans(acc), type = "b", pch = 19, col = "#2b5d9b")
legend("bottomright", c("mean test accuracy","5th-95th percentile of 50 draws"),
       col = c("#2b5d9b","#dfe7f1"), lwd = c(1,6), pch = c(19,NA), bty = "n", cex = .8)
summary_lc



<p class="caption"><b>Figura 8.</b> Exactitud media de prueba (línea) y su banda del percentil 5 al 95 sobre 50 muestras de entrenamiento aleatorias de cada tamaño. Con cuatro casos de entrenamiento el clasificador no es confiable &mdash; los peores sorteos quedan muy por debajo de la meseta &mdash; y la banda es ancha; con unas pocas decenas de casos la media ya alcanzó una meseta cercana a 0.93 y la banda se ha estrechado. Dos lecciones: una muestra de entrenamiento pequeña entrega un clasificador cuya calidad depende de la suerte, y más datos dejan de ayudar una vez que dominan las limitaciones propias del modelo (aquí, una frontera recta). Los modelos más ricos (Unidades 3, 4, 6) siguen mejorando durante más tiempo, y en consecuencia necesitan más datos.</p>

## 6. Aplicaciones en medicina y bioingeniería

Las aplicaciones del reconocimiento de patrones en salud son demasiadas para enumerarlas; esta sección selecciona ejemplos que ilustran las formas que adopta el problema y que están documentados en literatura primaria que puedes leer. Cada uno es un problema de clasificación o de predicción en el sentido de la Sección 4.

**La imagenología médica** es el campo en el que el aprendizaje profundo igualó por primera vez a los especialistas. Una red convolucional entrenada con 128,175 fotografías de retina detectó retinopatía diabética referible con una sensibilidad de 97&nbsp;% y una especificidad de 93&nbsp;% en un conjunto de validación, a la par de un panel de oftalmólogos;<sup class="c"><a href="#r16">[16]</a></sup> una red entrenada con 129,450 imágenes clínicas clasificó lesiones de piel con una exactitud del nivel de un dermatólogo;<sup class="c"><a href="#r17">[17]</a></sup> y el primer sistema de diagnóstico autónomo basado en IA &mdash; uno que emite una decisión sin que un clínico lea la imagen &mdash; fue autorizado por la FDA en 2018 para el tamizaje de retinopatía diabética. Hacia 2020, la FDA había aprobado más de 60 dispositivos médicos basados en IA, la mayoría en radiología.<sup class="c"><a href="#r18">[18]</a></sup> La Unidad 8 está dedicada a la clasificación de imágenes.

**Señales fisiológicas.** Una red entrenada con 91,232 registros de ECG de una sola derivación provenientes de 53,549 pacientes clasificó doce clases de ritmo con un desempeño superior al de cardiólogos individuales en un conjunto de prueba con etiquetas de consenso.<sup class="c"><a href="#r19">[19]</a></sup> El contexto de una sola derivación importa: es el que producen los dispositivos vestibles (wearables), y llevó la clasificación de ritmos del hospital a la muñeca.

**Predicción clínica a partir de la ficha clínica electrónica.** Los modelos que predicen deterioro, sepsis o reingreso a partir de variables registradas de forma rutinaria están hoy desplegados en muchos hospitales. Es también donde las fallas de la Sección 7 se han documentado con mayor claridad: una validación externa de un modelo propietario de sepsis ampliamente desplegado encontró una sensibilidad de 33&nbsp;% y un valor predictivo positivo de 12&nbsp;% con el umbral recomendado por el proveedor, muy por debajo del desempeño reportado durante su desarrollo.<sup class="c"><a href="#r20">[20]</a></sup> El modelo no estaba equivocado en sus propios datos; los hospitales eran distintos de aquellos en los que se desarrolló.

**Biología molecular y estructural** &mdash; el tema del curso de bioinformática. AlphaFold2 predice la estructura de proteínas a partir de la secuencia con una exactitud competitiva con la experimental;<sup class="c"><a href="#r12">[12]</a></sup> una red neuronal entrenada con la actividad antibacteriana de 2,335 moléculas, aplicada luego a una biblioteca de 6,111 moléculas en desarrollo clínico, identificó la halicina, un antibiótico estructuralmente novedoso activo contra patógenos resistentes;<sup class="c"><a href="#r21">[21]</a></sup> y los modelos QSAR que construiste en BGM324 son sistemas de reconocimiento de patrones cuyos objetos son moléculas.

**Lenguaje clínico.** Los grandes modelos de lenguaje responden preguntas médicas en el nivel de aprobación de los exámenes de habilitación o por encima de él,<sup class="c"><a href="#r4">[4]</a></sup> resumen fichas clínicas y redactan notas. Su modo de falla &mdash; afirmaciones fluidas, seguras y erróneas &mdash; es de un tipo nuevo, y su evaluación es un problema de investigación activo.

<div class="box key"><h4>Cómo leer un artículo de aplicación</h4><p>El Taller 2 del curso te pide presentar un artículo científico que aplique aprendizaje supervisado. La estructura de la Sección 4 es la guía de lectura: identifica los objetos, las características, las etiquetas y su procedencia, el tamaño y el origen de la muestra de entrenamiento, el clasificador, cómo se midió el desempeño y sobre qué casos (¿el mismo hospital?, ¿el mismo escáner?, ¿los mismos años?), y cuáles son los costos de los dos tipos de error. Un artículo que no responde estas preguntas no ha descrito un sistema de reconocimiento de patrones; lo ha publicitado.</p></div>

## 7. Datos, privacidad y ética

La descripción del curso menciona "la importancia de la protección de datos desde el punto de vista de la ética profesional" y "las regulaciones y los problemas éticos relacionados con la aplicación de la IA en salud". Estas son preocupaciones de ingeniería, no anexos: en una disciplina que aprende de datos sobre personas, el manejo de esos datos es parte del método. Esta sección trata cuatro problemas &mdash; la reidentificación, el cambio en la distribución de los datos, el sesgo de etiquetas y el marco regulatorio &mdash; y demuestra los tres primeros en código.

### 7.1 Datos personales, datos sensibles y desidentificación

Los datos de salud son <strong>datos personales sensibles</strong> en todo régimen moderno de protección de datos. En Chile, la Ley 19.628 de 1999 sobre protección de la vida privada rige los datos personales desde 1999, y la Ley 20.584 de 2012 establece la confidencialidad de la ficha clínica; la Ley 21.719, publicada en diciembre de 2024 y cuya entrada en vigencia está prevista para el 1 de diciembre de 2026, reescribe el régimen siguiendo las líneas del Reglamento General de Protección de Datos europeo (GDPR), crea una Agencia de Protección de Datos Personales con facultades de fiscalización y sanción, y clasifica los datos de salud y los datos genéticos como datos sensibles que requieren consentimiento explícito o una base legal específica.<sup class="c"><a href="#r22">[22]</a></sup> El GDPR (Reglamento (UE) 2016/679) es el régimen de referencia a nivel internacional, y la HIPAA Privacy Rule de los Estados Unidos (1996) define el estándar de desidentificación que sigue la mayoría de los conjuntos de datos médicos.

El remedio estándar para usar datos de salud en investigación es la <strong>desidentificación</strong>: eliminar los identificadores directos (nombre, número de identidad, dirección, teléfono). Hay dos términos que deben mantenerse separados. La <strong>anonimización</strong> es irreversible: nadie puede volver a vincular el registro con la persona. La <strong>seudonimización</strong> reemplaza los identificadores por un código y guarda la clave en otro lugar; es reversible para quien tenga la clave, y el GDPR considera que los datos seudonimizados siguen siendo personales. La mayoría de los conjuntos de datos clínicos "anonimizados" están en realidad seudonimizados, y muchos son reidentificables.

### 7.2 Reidentificación y k-anonimato

Eliminar los nombres no anonimiza un registro, porque las combinaciones de atributos corrientes &mdash; los <strong>cuasi-identificadores</strong> &mdash; suelen ser únicas. Latanya Sweeney mostró en 2000 que el 87&nbsp;% de la población de los Estados Unidos quedaba identificada de manera única por la tríada {código postal de 5 dígitos, fecha de nacimiento, sexo}, y lo demostró reidentificando la ficha médica del gobernador de Massachusetts en un conjunto de datos hospitalarios "desidentificado", al vincularlo con el padrón electoral público.<sup class="c"><a href="#r23">[23]</a></sup> Un estudio de 2019 estimó que el 99.98&nbsp;% de los estadounidenses sería reidentificado correctamente en cualquier conjunto de datos usando 15 atributos demográficos.<sup class="c"><a href="#r24">[24]</a></sup> El bloque de código de abajo reproduce el mecanismo en una población sintética y muestra la contramedida estándar, el <strong>k-anonimato</strong>: generalizar los cuasi-identificadores hasta que cada combinación sea compartida por al menos \( k \) registros.<sup class="c"><a href="#r23">[23]</a></sup>



In [ ]:
# Riesgo de reidentificación: ¿cuántos registros son únicos dados unos pocos cuasi-identificadores?
set.seed(0); N <- 20000
pop <- data.frame(
  birth_date = as.Date("1940-01-01") + sample(0:(365*65), N, replace = TRUE),
  sex        = sample(c("F","M"), N, replace = TRUE),
  comuna     = sample(sprintf("comuna_%02d", 0:59), N, replace = TRUE),   # 60 comunas
  diagnosis  = sample(c("hypertension","diabetes","asthma","depression","HIV"), N,
                      replace = TRUE, prob = c(.4,.3,.15,.1,.05)))
share_unique <- function(cols) {
  key <- do.call(paste, pop[cols]); tt <- table(key)
  c(unique_frac = sum(tt[key] == 1) / N, k_min = min(tt)) }
for (cols in list("sex", c("sex","comuna"), c("sex","comuna","birth_date"))) {
  r <- share_unique(cols)
  cat(sprintf("%-34s -> unique %5.1f%%   (smallest group k = %d)\n",
              paste(cols, collapse = "+"), 100 * r["unique_frac"], r["k_min"])) }


In [ ]:
# k-anonimato por generalización: reemplazar la fecha de nacimiento exacta por el año y luego por una banda de 10 años.
pop$birth_year  <- as.integer(format(pop$birth_date, "%Y"))
pop$birth_decade <- (pop$birth_year %/% 10) * 10
for (cols in list(c("sex","comuna","birth_year"), c("sex","comuna","birth_decade"))) {
  r <- share_unique(cols)
  cat(sprintf("%-34s -> unique %5.1f%%   (smallest group k = %d)\n",
              paste(cols, collapse = "+"), 100 * r["unique_frac"], r["k_min"])) }



Con el sexo solo, nadie es identificable (el grupo más pequeño reúne a miles de personas); con el sexo y la comuna, todavía nadie, porque 20,000 personas se reparten en 120 celdas. Agregar la fecha de nacimiento exacta hace que el 99.4&nbsp;% de los registros sean únicos &mdash; un atacante que conoce el cumpleaños, el sexo y la comuna de un vecino puede encontrar su diagnóstico. Generalizar la fecha al año deja un 7.6&nbsp;% residual de registros únicos; generalizarla a la década logra k-anonimato con un \( k \) muy superior a 1 (el grupo más pequeño reúne a seis personas), a costa de información: un modelo que necesitaba la edad en años la ha perdido. Ese compromiso entre utilidad y privacidad es inevitable, y le corresponde al ingeniero asumirlo explícitamente, junto con el dueño de los datos. Observa también que el k-anonimato protege la identidad, no los atributos: si todos los registros de un grupo tienen el mismo diagnóstico, saber que alguien pertenece al grupo revela su diagnóstico (el problema que abordan las nociones más fuertes de <em>l-diversidad</em> y <em>privacidad diferencial</em>, que están fuera del alcance de este curso).

<div class="box key"><h4>Ejercicio 3</h4><p>Reduce la población a <code>N = 2000</code> &mdash; un estudio realista de un solo hospital &mdash; y repite ambos bloques de código. ¿Qué generalización se necesita ahora para que cada registro comparta sus cuasi-identificadores con al menos otros cinco? Los conjuntos de datos pequeños son más reidentificables, no menos.</p></div>

### 7.3 Cambio en la distribución de los datos y aprendizaje por atajos

Un clasificador aprende cualquier regularidad de la muestra de entrenamiento que prediga la etiqueta, tenga o no relación con la enfermedad. Cuando una regularidad es un accidente de la forma en que se recolectaron los datos, el clasificador ha aprendido un <strong>atajo</strong> (shortcut), y falla en cuanto los datos se recolectan de otra manera.<sup class="c"><a href="#r25">[25]</a></sup> El caso documentado es instructivo: una red entrenada para detectar neumonía en radiografías de tórax de varios hospitales aprendió a reconocer <em>de qué hospital</em> provenía una imagen &mdash; por la posición de marcadores metálicos y el estilo de la imagen &mdash; y usó la prevalencia de neumonía de ese hospital como sustituto, logrando una alta exactitud interna y una exactitud externa deficiente.<sup class="c"><a href="#r26">[26]</a></sup> La simulación de abajo reproduce el mecanismo con los datos de cáncer de mama: se agrega un "identificador de escáner" que, en el hospital de entrenamiento, resulta estar correlacionado con la etiqueta porque los casos sospechosos se derivaban a una misma máquina &mdash; un artefacto plausible del flujo de trabajo.



In [ ]:
# Aprendizaje por atajos: una característica espuria que predice la etiqueta en el entrenamiento pero no en el despliegue.
set.seed(0)
tri <- sample(nrow(bc), round(0.7 * nrow(bc)))
Xtr <- X[tri, ]; Xte <- X[-tri, ]; ytr <- y[tri]; yte <- y[-tri]
mal_tr <- ytr == "malignant"; mal_te <- yte == "malignant"
# Hospital de entrenamiento: el 97% de los casos malignos en el escáner 1 y el 97% de los benignos en el escáner 0.
scanner_tr   <- ifelse(runif(length(ytr)) < 0.97, as.integer(mal_tr), as.integer(!mal_tr))
scanner_same <- ifelse(runif(length(yte)) < 0.97, as.integer(mal_te), as.integer(!mal_te))  # mismo hospital
scanner_diff <- sample(0:1, length(yte), replace = TRUE)                                    # hospital nuevo
ctl <- rpart.control(maxdepth = 3, cp = 0, xval = 0)
tree_shortcut <- rpart(lab ~ ., data.frame(Xtr, scanner = scanner_tr, lab = ytr), method = "class", control = ctl)
tree_honest   <- rpart(lab ~ ., data.frame(Xtr, lab = ytr), method = "class", control = ctl)
cat("Root split of the model with the scanner feature:", as.character(tree_shortcut$frame$var[1]), "\n\n")
acc <- function(m, d, truth) mean(predict(m, d, type = "class") == truth)
data.frame(
  `training accuracy`            = c(acc(tree_shortcut, data.frame(Xtr, scanner = scanner_tr), ytr), acc(tree_honest, Xtr, ytr)),
  `new cases, same hospital`     = c(acc(tree_shortcut, data.frame(Xte, scanner = scanner_same), yte), acc(tree_honest, Xte, yte)),
  `new cases, different hospital`= c(acc(tree_shortcut, data.frame(Xte, scanner = scanner_diff), yte), acc(tree_honest, Xte, yte)),
  row.names = c("model WITH scanner feature", "model WITHOUT scanner feature"), check.names = FALSE) |> round(3)



El modelo al que se le dio el identificador de escáner lo puso en la raíz de su árbol: es la característica más predictiva por sí sola en el hospital de entrenamiento. Su exactitud es excelente en los datos de entrenamiento y en los casos nuevos del mismo hospital &mdash; allí el atajo sigue funcionando &mdash; y se desploma en el hospital nuevo (de alrededor de 0.98 a 0.88), donde el escáner no aporta información. El modelo sin la característica espuria prácticamente no cambia entre hospitales. Nada en las cifras del hospital de entrenamiento advertía de la falla; solo la prueba con datos del entorno de despliegue la reveló. Esta es la lección general de la validación externa, y la razón por la que falló el modelo de sepsis de la Sección 6.<sup class="c"><a href="#r20">[20]</a></sup>

### 7.4 Sesgo de etiquetas: el algoritmo aprende la etiqueta, no la intención

En 2019, Obermeyer y colaboradores analizaron un algoritmo comercial usado por sistemas de salud que cubrían a unos 200 millones de personas en los Estados Unidos para identificar a pacientes con necesidades complejas que requerían atención adicional.<sup class="c"><a href="#r27">[27]</a></sup> El algoritmo predecía el <em>costo de atención de salud</em> futuro como sustituto de la <em>necesidad de salud</em>. Como se gasta menos dinero en pacientes negros que en pacientes blancos con el mismo nivel de enfermedad &mdash; por razones de acceso, confianza y discriminación &mdash;, el algoritmo clasificaba a los pacientes negros como de menor riesgo que a pacientes blancos igualmente enfermos: para un mismo puntaje de riesgo, los pacientes negros tenían en promedio 26.3&nbsp;% más enfermedades crónicas. No se usó ninguna variable racial; el sesgo entró por la elección de la etiqueta. Reformular la etiqueta como una medida de enfermedad en lugar de costo redujo la disparidad en un 84&nbsp;%. La simulación de abajo reproduce el mecanismo en su forma más simple.



In [ ]:
# Sesgo de etiquetas: dos grupos con la misma enfermedad y gasto desigual; un modelo entrenado con el costo los ordena de forma distinta.
set.seed(0); n_per <- 5000
illness <- rgamma(2 * n_per, shape = 2, scale = 1)          # necesidad de salud real, misma distribución en ambos grupos
group   <- rep(c("A","B"), each = n_per)
spend   <- ifelse(group == "A", 1.0, 0.7)                   # el grupo B recibe 30% menos atención por la misma enfermedad
cost    <- illness * spend * rlnorm(2 * n_per, 0, 0.3)      # costo observado (la etiqueta con la que se entrena el algoritmo)
flagged <- cost >= quantile(cost, 0.8)                      # marcar el 20% superior según el costo predicho
tab <- data.frame(
  mean_illness            = tapply(illness, group, mean),
  mean_cost               = tapply(cost, group, mean),
  share_flagged           = tapply(flagged, group, mean),
  mean_illness_of_flagged = tapply(illness[flagged], group[flagged], mean))
round(tab, 3)



Los dos grupos están, por construcción, igualmente enfermos (necesidad de salud media de alrededor de 2.0 en ambos). Como uno recibe menos atención, su costo registrado es menor, y un algoritmo entrenado para predecir el costo &mdash; perfectamente, en esta simulación &mdash; marca al 27&nbsp;% del grupo A pero solo al 13&nbsp;% del grupo B, y los miembros del grupo B que sí marca están <em>más enfermos</em> que los marcados en el grupo A (enfermedad 4.4 frente a 3.7). El algoritmo es exacto en su tarea; la tarea era la equivocada. Elegir la etiqueta es la decisión de modelado con mayores consecuencias y rara vez se trata como tal.

### 7.5 El marco regulatorio y ético

Tres cuerpos de normas afectan a un ingeniero que construye sistemas de aprendizaje para la salud, y el curso espera que sepas qué cubre cada uno.

**La protección de datos** (Sección 7.1): las Leyes chilenas 19.628 y 20.584, la nueva Ley 21.719,<sup class="c"><a href="#r22">[22]</a></sup> el GDPR y la HIPAA rigen la <em>recolección y el tratamiento</em> de los datos personales &mdash; base de licitud, consentimiento, limitación de la finalidad, minimización, los derechos del titular de los datos y la obligación de notificar las vulneraciones de seguridad.

**La regulación de dispositivos médicos** rige el <em>sistema</em>. El software que informa o toma una decisión clínica es un dispositivo médico (software como dispositivo médico, SaMD). En los Estados Unidos lo regula la FDA, que ha publicado un plan de acción y principios de buenas prácticas de aprendizaje automático para dispositivos basados en IA;<sup class="c"><a href="#r18">[18]</a></sup> en la Unión Europea se aplica el Reglamento de Dispositivos Médicos (2017/745) y, desde 2024, la <strong>Ley de Inteligencia Artificial</strong> (Artificial Intelligence Act) clasifica la IA usada en dispositivos médicos como de <em>alto riesgo</em>, con obligaciones de gestión de riesgos, gobernanza de datos, documentación, supervisión humana, exactitud y robustez.<sup class="c"><a href="#r28">[28]</a></sup> En Chile, el Instituto de Salud Pública (ISP) registra los dispositivos médicos, y desde 2021 se encuentran en desarrollo una política nacional de IA y un proyecto de ley.

**Las orientaciones éticas** rigen la <em>práctica</em>. La guía de 2021 de la Organización Mundial de la Salud sobre la ética y la gobernanza de la IA para la salud establece seis principios &mdash; proteger la autonomía; promover el bienestar humano, la seguridad y el interés público; garantizar la transparencia, la explicabilidad y la inteligibilidad; fomentar la responsabilidad y la rendición de cuentas; garantizar la inclusividad y la equidad; y promover una IA receptiva y sostenible.<sup class="c"><a href="#r29">[29]</a></sup> No son consignas; cada uno se corresponde con un requisito de ingeniería demostrado en esta sección: la equidad, con el análisis del sesgo de etiquetas de 7.4; la robustez, con la validación externa de 7.3; la transparencia, con las reglas legibles de la Sección 3.5; y la autonomía y la privacidad, con la desidentificación de 7.2.

### 7.6 Contenido generado, desinformación y el debate público sobre el riesgo

La clase cierra con un ejercicio: pares de rostros, algunos fotografiados y otros generados, y la pregunta "¿cuál es auténtico?". La mayoría de las personas acierta al nivel del azar o cerca de él, y los modelos han mejorado desde que se hicieron las diapositivas. Los medios sintéticos que muestran a personas reales diciendo o haciendo cosas que no dijeron ni hicieron &mdash; los <strong>deepfakes</strong> &mdash; se identificaron como una amenaza para la privacidad, la deliberación democrática y la seguridad tan pronto como los métodos generativos se volvieron prácticos,<sup class="c"><a href="#r37">[37]</a></sup> y el punto relevante para un ingeniero de salud es que las mismas técnicas generan imágenes médicas sintéticas, notas clínicas sintéticas y voces sintéticas, cada una de las cuales puede usarse para investigación (aumentar un conjunto de entrenamiento, proteger la privacidad) o para fraude (fabricar evidencia de una enfermedad ante un seguro, suplantar a un clínico). La detección es una carrera armamentista que van ganando los generadores; la procedencia &mdash; la firma criptográfica del contenido auténtico en el momento de su captura &mdash; es la dirección que está tomando la regulación.

El debate público sobre el riesgo de la IA abarca un rango amplio, desde los daños concretos documentados en las Secciones 7.2&ndash;7.5 hasta declaraciones de líderes de laboratorios, incluido el director ejecutivo de OpenAI citado en la clase, según las cuales la IA avanzada podría plantear riesgos de la escala de una pandemia o de una guerra nuclear. Este curso no zanja ese debate. Te pide notar que los daños concretos son los que estarás en posición de causar o de prevenir en tu vida profesional, que se comprenden bien y que cada uno tiene una contramedida de ingeniería que ya has visto en código.

<div class="box warn"><h4>La obligación del profesional</h4><p>Un sistema que tiene un 95&nbsp;% de exactitud en sus datos de desarrollo y que se despliega sin validación externa, con datos obtenidos sin una base de licitud y con una etiqueta que codifica una inequidad, cumple todos los criterios técnicos de esta unidad y viola todos los éticos. Los métodos de este curso te enseñan a construir tales sistemas; esta sección te dice que construirlos bien incluye negarse a desplegarlos antes de haber respondido las preguntas de las Secciones 7.1&ndash;7.4. El Taller 1 del curso te pide encontrar y presentar un caso en el que la IA pone en peligro alguna faceta de la vida humana; los mecanismos de esta sección &mdash; reidentificación, aprendizaje por atajos, sesgo de etiquetas &mdash; son el lugar donde buscar.</p></div>

## 8. Resumen

Ahora deberías poder explicar, sin apuntes:

- Por qué la "inteligencia artificial" no tiene una definición única, y qué ofrece a un ingeniero la definición del agente racional que la prueba de Turing no ofrece.
- La secuencia McCulloch&ndash;Pitts &rarr; perceptrón &rarr; <em>Perceptrons</em> &rarr; retropropagación &rarr; AlexNet, qué aportó cada etapa y por qué ocurrieron los dos inviernos.
- La diferencia entre el conocimiento que se programa (sistemas expertos como MYCIN) y el conocimiento que se aprende, y por qué este último convirtió los datos en la preocupación central del campo.
- Los cuatro enfoques &mdash; simbólico, conexionista, estadístico, evolutivo &mdash; con un ejemplo médico de cada uno; la diferencia entre un algoritmo y un modelo; los niveles ANI/AGI/ASI y la taxonomía reactivo / memoria limitada / teoría de la mente / autoconsciente, y dónde se ubica cada sistema de este curso (memoria limitada).
- Qué es la IA generativa (texto, imagen, video), por qué GPT es generativo y BERT no lo es, y qué nuevos modos de falla trae (falsedades fluidas; medios sintéticos).
- Las cinco etapas del pipeline de reconocimiento de patrones y en cuáles dos se concentra este curso.
- La matriz de datos \( \mathbf{X} \) (\( n \times p \)) y el vector de etiquetas \( \mathbf{y} \); los cinco tipos de datos y por qué las variables nominales deben codificarse con one-hot.
- Cómo convertir una imagen o una señal en un vector de características, y por qué un patrón es una región del espacio de características.
- El clasificador del centroide más cercano: cómo se entrena, cómo decide, qué aspecto tiene su frontera de decisión y cómo depende su exactitud del tamaño de la muestra de entrenamiento.
- Las tres propiedades de la muestra de entrenamiento (muestra &ne; población; las etiquetas definen la tarea; el tamaño limita el aprendizaje) y el modo de falla asociado a cada una (cambio en la distribución de los datos y atajos; sesgo de etiquetas; sobreajuste).
- La reidentificación mediante cuasi-identificadores, el k-anonimato y las normas chilenas e internacionales que rigen los datos de salud y los dispositivos médicos con IA.

## 9. Autoevaluación

<details><summary>1. Las cuatro definiciones de IA de Russell y Norvig difieren a lo largo de dos ejes. Nómbralos y di en qué cuadrante se ubica un sistema de tamizaje de retina y por qué.</summary>Humano frente a racional, y pensamiento frente a acción. Un sistema de tamizaje es un agente racional (actuar racionalmente): percibe una imagen y actúa &mdash; entrega una decisión de derivación &mdash; de modo de maximizar un objetivo medible; no pretende pensar como un humano ni parecer uno.</details>

<details><summary>2. ¿Qué demostraron Minsky y Papert en 1969, y por qué eso no impidió las redes neuronales actuales?</summary>Que un perceptrón de una sola capa no puede representar funciones como XOR, que no son linealmente separables. Las redes multicapa sí pueden representarlas, y la retropropagación (1986) hizo entrenables las redes multicapa; lo que faltaba en 1969 era un algoritmo de entrenamiento y el cómputo para ejecutarlo, no una posibilidad teórica.</details>

<details><summary>3. MYCIN alcanzó un desempeño de nivel especialista y nunca se desplegó. Da dos razones y di cuál sigue aplicándose hoy a los sistemas de IA.</summary>La responsabilidad legal por las recomendaciones de una máquina, la integración en el flujo de trabajo clínico (las sesiones interactivas tomaban más tiempo que una interconsulta), la infraestructura computacional de los años 1970, y la fragilidad y el costo de mantenimiento de las reglas escritas a mano. La responsabilidad legal y la integración en el flujo de trabajo siguen siendo hoy las principales barreras; la fragilidad se ha transformado más que resuelto (los sistemas que aprenden son frágiles ante el cambio en la distribución de los datos).</details>

<details><summary>4. En los bloques de código de la Sección 3.5, el árbol aprendido (95.5%) concuerda con la biopsia más a menudo que la regla escrita a mano (89.0%). Da dos razones por las que esa comparación es injusta con la regla escrita a mano.</summary>La concordancia del árbol se midió sobre los mismos casos con los que se ajustó, por lo que tiene un sesgo optimista; y el árbol exploró las nueve características y todos los umbrales, mientras que la regla usó dos características con umbrales de números redondos. Una comparación justa evaluaría ambas sobre casos reservados (Suplemento S1).</details>

<details><summary>5. Un conjunto de datos codifica el estadio tumoral como I = 1, II = 2, III = 3, IV = 4 y el grupo sanguíneo como O = 1, A = 2, B = 3, AB = 4. ¿Qué codificación es defendible y cuál no, y qué debería hacerse en su lugar?</summary>El estadio es ordinal, de modo que los códigos enteros preservan el orden y son defendibles si se acepta un espaciado uniforme. El grupo sanguíneo es nominal; la codificación entera inventa un orden y distancias. Debería codificarse con one-hot en cuatro columnas binarias.</details>

<details><summary>6. En la Figura 5, las señales de 1,000 muestras se redujeron a dos características. Nombra una información que se pierde en esa reducción y describe cómo averiguarías si la pérdida importa.</summary>Se pierde la forma de onda de cada latido (por ejemplo, el ancho del QRS o la morfología de la onda T); solo se conserva la temporización. Si importa se decide empíricamente: entrenar clasificadores con ambas representaciones y comparar su desempeño sobre datos reservados para la tarea en cuestión.</details>

<details><summary>7. Calcula a mano: clasificador del centroide más cercano con centroides \( \mu_B = (12, 2) \) y \( \mu_M = (19, 9) \); clasifica \( \mathbf{x} = (16, 6) \).</summary>\( d_B = \sqrt{16 + 16} = \sqrt{32} \approx 5.66 \); \( d_M = \sqrt{9 + 9} = \sqrt{18} \approx 4.24 \). El caso se clasifica como maligno. (Está del lado maligno de la mediatriz, que pasa por el punto medio (15.5, 5.5)).</details>

<details><summary>8. En la Figura 8, la exactitud deja de mejorar después de unas pocas decenas de casos de entrenamiento. ¿Es esto evidencia de que más datos serían inútiles para este problema?</summary>No. Es evidencia de que más datos serían inútiles para <em>este clasificador</em>, cuya frontera rectilínea no puede representar la superposición entre las clases. Un clasificador más flexible seguiría beneficiándose de más datos (y los necesitaría para evitar el sobreajuste).</details>

<details><summary>9. Un hospital publica un conjunto de datos del que se eliminaron el nombre y el número de identidad, conservando la fecha de nacimiento, el sexo y la comuna. ¿Está anonimizado? ¿Cuál es el cambio mínimo que lo haría 5-anónimo, y qué costo tiene?</summary>No; esos tres cuasi-identificadores hacen que la mayoría de los registros sean únicos (Sección 7.2). Generalizar la fecha de nacimiento al año o a la década, y posiblemente la comuna a la región, hasta que cada combinación aparezca al menos cinco veces lograría el 5-anonimato, a costa de la resolución de la edad y del detalle geográfico para cualquier modelo posterior.</details>

<details><summary>10. En la simulación de aprendizaje por atajos, ¿qué número por sí solo habría revelado el problema antes del despliegue, y por qué las cifras del hospital de entrenamiento no lo revelan?</summary>La exactitud sobre datos del hospital de despliegue (validación externa). En el hospital de entrenamiento, el identificador de escáner realmente predice la etiqueta, de modo que toda medición interna &mdash; exactitud de entrenamiento, exactitud sobre casos reservados del mismo hospital &mdash; se mantiene alta.</details>

<details><summary>11. El algoritmo de Obermeyer et al. no usaba ninguna variable racial. ¿Cómo entró en él el sesgo racial, y cuál fue el remedio?</summary>A través de la etiqueta: el costo de atención de salud se usó como sustituto de la necesidad de salud, y el costo es sistemáticamente menor para los pacientes negros con el mismo nivel de enfermedad. Reemplazar la etiqueta de costo por una medida de enfermedad (número de enfermedades crónicas activas) redujo la disparidad en un 84&nbsp;%.</details>

<details><summary>12. "El algoritmo decidió rechazar la solicitud de reembolso." Reescribe la frase usando el vocabulario de la Sección 1.2 y explica qué cambió.</summary>"Un modelo, producido por un algoritmo de aprendizaje a partir de datos históricos de solicitudes de reembolso, predijo una alta probabilidad de fraude y el sistema actuó según esa predicción." La reescritura hace explícito que el comportamiento se aprendió de los datos (de modo que se pueden examinar la muestra de entrenamiento y su etiqueta), que un umbral y una política de acción fueron elegidos por personas, y que el objeto desplegado es un modelo, no un procedimiento.</details>

<details><summary>13. Deep Blue se clasifica como "reactivo" y un filtro de spam entrenado como de "memoria limitada", aunque Deep Blue exploraba millones de posiciones y el filtro de spam solo cuenta palabras. ¿Por qué el filtro se ubica en el nivel superior?</summary>La taxonomía ordena según el uso del pasado, no según el esfuerzo computacional. La evaluación de Deep Blue quedó fija en el momento del diseño y no llevaba nada de una partida a la siguiente; los parámetros del filtro se aprendieron de ejemplos pasados almacenados y cambian si se reentrena. Aprender de la experiencia almacenada es lo que define el nivel de memoria limitada.</details>

<details><summary>14. ¿A cuál de los seis principios de la OMS sirve la validación externa, y a cuál sirve el k-anonimato?</summary>La validación externa sirve a la seguridad, a la robustez (bienestar e interés público) y, cuando se estratifica por subgrupos, a la inclusividad y la equidad. El k-anonimato sirve a la protección de la autonomía y de la privacidad.</details>

## Glosario

<dl class="gloss">
<dt>Inteligencia artificial (artificial intelligence, AI)</dt><dd>El estudio y la construcción de sistemas que perciben su entorno y actúan de modo de lograr sus objetivos; históricamente, el intento de que las máquinas realicen tareas que requieren inteligencia cuando las realizan personas.</dd>
<dt>Anonimización / seudonimización (anonymisation / pseudonymisation)</dt><dd>Eliminación irreversible del vínculo entre un registro y una persona / reemplazo de los identificadores por un código cuya clave se guarda por separado (reversible; siguen siendo datos personales según el GDPR).</dd>
<dt>Clasificador (classifier)</dt><dd>Una función que asigna a un vector de características una etiqueta de clase o una distribución de probabilidad sobre las clases; sus parámetros se aprenden de una muestra de entrenamiento.</dd>
<dt>Conexionismo (connectionism)</dt><dd>El enfoque de la IA en el que el conocimiento se almacena de forma implícita en los pesos de redes de unidades simples y se adquiere mediante entrenamiento.</dd>
<dt>Cambio en la distribución de los datos (dataset shift)</dt><dd>Una diferencia entre la distribución de los datos de entrenamiento y la de los datos encontrados en el despliegue; la causa principal de la degradación del desempeño de los modelos desplegados.</dd>
<dt>Deepfake (deepfake)</dt><dd>Imagen, audio o video sintético que muestra a una persona real diciendo o haciendo algo que no dijo ni hizo; producido por modelos generativos.</dd>
<dt>Efecto ELIZA (ELIZA effect)</dt><dd>La tendencia de los usuarios a atribuir comprensión o intención a un sistema que solo manipula símbolos; debe su nombre al chatbot de Weizenbaum de 1966.</dd>
<dt>Sistema experto (expert system)</dt><dd>Un programa de IA simbólica que codifica el conocimiento de un experto humano como reglas y razona con ellas; MYCIN es el prototipo médico.</dd>
<dt>IA generativa (generative AI)</dt><dd>Modelos entrenados con grandes colecciones de contenido que producen contenido nuevo del mismo tipo (texto, imágenes, video, moléculas); los grandes modelos de lenguaje y los modelos de difusión son las familias principales.</dd>
<dt>Característica / vector de características (feature / feature vector)</dt><dd>Una propiedad medida o calculada de un objeto / el vector de longitud fija con todas esas propiedades, que representa al objeto ante el clasificador.</dd>
<dt>k-anonimato (k-anonymity)</dt><dd>La propiedad de un conjunto de datos en el que cada combinación de valores de los cuasi-identificadores es compartida por al menos k registros.</dd>
<dt>Etiqueta (label, target, response)</dt><dd>La clase o el valor conocido asociado a cada objeto de entrenamiento; lo que el clasificador aprende a predecir.</dd>
<dt>Aprendizaje automático (machine learning)</dt><dd>El subcampo de la IA en el que el comportamiento de un sistema se aprende a partir de datos en lugar de programarse.</dd>
<dt>Modelo frente a algoritmo (model vs algorithm)</dt><dd>Un algoritmo es un procedimiento fijo; un modelo es un objeto con parámetros aprendidos de los datos por un algoritmo de aprendizaje. Lo que se despliega es el modelo.</dd>
<dt>Clasificador del centroide más cercano (nearest-centroid classifier)</dt><dd>El clasificador que asigna un objeto a la clase cuyo vector de características medio está más cerca; sus fronteras de decisión son hiperplanos.</dd>
<dt>Nominal / ordinal / continuo (nominal / ordinal / continuous)</dt><dd>Tipos de datos: categorías no ordenadas / categorías ordenadas sin espaciado fijo / mediciones con valores reales. El tipo determina las operaciones admisibles.</dd>
<dt>Codificación one-hot (one-hot encoding)</dt><dd>Representación de una variable nominal con k niveles como k columnas indicadoras binarias, de modo que no se implique ningún orden ni distancia entre los niveles.</dd>
<dt>Reconocimiento de patrones (pattern recognition)</dt><dd>La disciplina que se ocupa de la clasificación automática de objetos (imágenes, señales, mediciones) en categorías.</dd>
<dt>Perceptrón (perceptron)</dt><dd>La máquina que aprendía de Rosenblatt, de 1958: una unidad de umbral cuyos pesos se ajustan a partir de ejemplos etiquetados; el antecesor de las redes neuronales.</dd>
<dt>Cuasi-identificador (quasi-identifier)</dt><dd>Un atributo que no es un identificador por sí mismo pero que, combinado con otros, puede singularizar a una persona (fecha de nacimiento, sexo, código postal).</dd>
<dt>Agente racional (rational agent)</dt><dd>Un sistema que percibe mediante sensores y actúa mediante actuadores de modo de maximizar el logro esperado de su objetivo; la definición de trabajo de la IA en los libros de texto modernos.</dd>
<dt>Aprendizaje por atajos (shortcut learning)</dt><dd>La dependencia de un modelo de una regularidad espuria de los datos de entrenamiento (como un marcador del hospital) que predice la etiqueta allí pero no en otros lugares.</dd>
<dt>Muestra de entrenamiento (training sample, training set)</dt><dd>Los datos etiquetados a partir de los cuales se aprenden los parámetros de un clasificador; una muestra de la población a la que se aplicará el clasificador, no su totalidad.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">McCarthy J, Minsky ML, Rochester N, Shannon CE. A proposal for the Dartmouth Summer Research Project on Artificial Intelligence, August 31, 1955. <em>AI Magazine</em>. 2006;27(4):12&ndash;14. <a href="https://doi.org/10.1609/aimag.v27i4.1904">doi:10.1609/aimag.v27i4.1904</a></li>
<li id="r2">Turing AM. Computing machinery and intelligence. <em>Mind</em>. 1950;LIX(236):433&ndash;460. <a href="https://doi.org/10.1093/mind/LIX.236.433">doi:10.1093/mind/LIX.236.433</a></li>
<li id="r3">Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. Capítulo 1. <a href="https://doi.org/10.1109/MIS.2022.3186559">doi:10.1109/MIS.2022.3186559</a> (reseña de la 4.ª edición en <em>IEEE Intelligent Systems</em>; el libro en sí no tiene DOI)</li>
<li id="r4">Singhal K, Azizi S, Tu T, et al. Large language models encode clinical knowledge. <em>Nature</em>. 2023;620:172&ndash;180. <a href="https://doi.org/10.1038/s41586-023-06291-2">doi:10.1038/s41586-023-06291-2</a></li>
<li id="r5">McCulloch WS, Pitts W. A logical calculus of the ideas immanent in nervous activity. <em>Bull Math Biophys</em>. 1943;5:115&ndash;133. <a href="https://doi.org/10.1007/BF02478259">doi:10.1007/BF02478259</a></li>
<li id="r6">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386&ndash;408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r7">Minsky M, Papert S. <em>Perceptrons: An Introduction to Computational Geometry</em>. MIT Press; 1969. Reeditado en 2017. <a href="https://doi.org/10.7551/mitpress/11301.001.0001">doi:10.7551/mitpress/11301.001.0001</a></li>
<li id="r8">Shortliffe EH, Buchanan BG. A model of inexact reasoning in medicine. <em>Math Biosci</em>. 1975;23(3&ndash;4):351&ndash;379. <a href="https://doi.org/10.1016/0025-5564(75)90047-4">doi:10.1016/0025-5564(75)90047-4</a></li>
<li id="r9">Rumelhart DE, Hinton GE, Williams RJ. Learning representations by back-propagating errors. <em>Nature</em>. 1986;323:533&ndash;536. <a href="https://doi.org/10.1038/323533a0">doi:10.1038/323533a0</a></li>
<li id="r10">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84&ndash;90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r11">Silver D, Huang A, Maddison CJ, et al. Mastering the game of Go with deep neural networks and tree search. <em>Nature</em>. 2016;529:484&ndash;489. <a href="https://doi.org/10.1038/nature16961">doi:10.1038/nature16961</a></li>
<li id="r12">Jumper J, Evans R, Pritzel A, et al. Highly accurate protein structure prediction with AlphaFold. <em>Nature</em>. 2021;596:583&ndash;589. <a href="https://doi.org/10.1038/s41586-021-03819-2">doi:10.1038/s41586-021-03819-2</a></li>
<li id="r13">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861&ndash;870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r14">Jain AK, Duin RPW, Mao J. Statistical pattern recognition: a review. <em>IEEE Trans Pattern Anal Mach Intell</em>. 2000;22(1):4&ndash;37. <a href="https://doi.org/10.1109/34.824819">doi:10.1109/34.824819</a></li>
<li id="r15">Alpaydin E, Kaynak C. Cascading classifiers. <em>Kybernetika</em>. 1998;34(4):369&ndash;374. Conjunto de datos UCI Optical Recognition of Handwritten Digits. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r16">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402&ndash;2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r17">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115&ndash;118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r18">Benjamens S, Dhunnoo P, Meskó B. The state of artificial intelligence-based FDA-approved medical devices and algorithms: an online database. <em>npj Digit Med</em>. 2020;3:118. <a href="https://doi.org/10.1038/s41746-020-00324-0">doi:10.1038/s41746-020-00324-0</a></li>
<li id="r19">Hannun AY, Rajpurkar P, Haghpanahi M, et al. Cardiologist-level arrhythmia detection and classification in ambulatory electrocardiograms using a deep neural network. <em>Nat Med</em>. 2019;25:65&ndash;69. <a href="https://doi.org/10.1038/s41591-018-0268-3">doi:10.1038/s41591-018-0268-3</a></li>
<li id="r20">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065&ndash;1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r21">Stokes JM, Yang K, Swanson K, et al. A deep learning approach to antibiotic discovery. <em>Cell</em>. 2020;180(4):688&ndash;702. <a href="https://doi.org/10.1016/j.cell.2020.01.021">doi:10.1016/j.cell.2020.01.021</a></li>
<li id="r22">Ley N° 21.719, que regula la protección y el tratamiento de los datos personales y crea la Agencia de Protección de Datos Personales. Diario Oficial de la República de Chile, 13 de diciembre de 2024; entrada en vigencia el 1 de diciembre de 2026. Biblioteca del Congreso Nacional. <a href="https://www.bcn.cl/leychile">https://www.bcn.cl/leychile</a> (buscar "Ley 21719"; allí también está disponible el texto refundido de la Ley 19.628)</li>
<li id="r23">Sweeney L. k-anonymity: a model for protecting privacy. <em>Int J Uncertain Fuzziness Knowl-Based Syst</em>. 2002;10(5):557&ndash;570. <a href="https://doi.org/10.1142/S0218488502001648">doi:10.1142/S0218488502001648</a></li>
<li id="r24">Rocher L, Hendrickx JM, de Montjoye Y-A. Estimating the success of re-identifications in incomplete datasets using generative models. <em>Nat Commun</em>. 2019;10:3069. <a href="https://doi.org/10.1038/s41467-019-10933-3">doi:10.1038/s41467-019-10933-3</a></li>
<li id="r25">Geirhos R, Jacobsen J-H, Michaelis C, et al. Shortcut learning in deep neural networks. <em>Nat Mach Intell</em>. 2020;2:665&ndash;673. <a href="https://doi.org/10.1038/s42256-020-00257-z">doi:10.1038/s42256-020-00257-z</a></li>
<li id="r26">Zech JR, Badgeley MA, Liu M, Costa AB, Titano JJ, Oermann EK. Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. <em>PLoS Med</em>. 2018;15(11):e1002683. <a href="https://doi.org/10.1371/journal.pmed.1002683">doi:10.1371/journal.pmed.1002683</a></li>
<li id="r27">Obermeyer Z, Powers B, Vogeli C, Mullainathan S. Dissecting racial bias in an algorithm used to manage the health of populations. <em>Science</em>. 2019;366(6464):447&ndash;453. <a href="https://doi.org/10.1126/science.aax2342">doi:10.1126/science.aax2342</a></li>
<li id="r28">Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence (Artificial Intelligence Act). <em>Official Journal of the European Union</em>. 2024;L 2024/1689. <a href="https://eur-lex.europa.eu/eli/reg/2024/1689/oj">https://eur-lex.europa.eu/eli/reg/2024/1689/oj</a></li>
<li id="r29">World Health Organization. <em>Ethics and governance of artificial intelligence for health: WHO guidance</em>. Geneva: WHO; 2021. ISBN 978-92-4-002920-0. <a href="https://www.who.int/publications/i/item/9789240029200">https://www.who.int/publications/i/item/9789240029200</a></li>
<li id="r30">Hintze A. Understanding the four types of AI, from reactive robots to self-aware beings. <em>The Conversation</em>. 14 November 2016. <a href="https://theconversation.com/understanding-the-four-types-of-ai-from-reactive-robots-to-self-aware-beings-67616">theconversation.com/&hellip;-67616</a> (artículo de divulgación científica; la taxonomía no tiene una fuente de investigación primaria)</li>
<li id="r31">Hebb DO. <em>The Organization of Behavior: A Neuropsychological Theory</em>. New York: Wiley; 1949. Para la formulación moderna de la regla, véase Brown TH, Kairiss EW, Keenan CL. Hebbian synapses: biophysical mechanisms and algorithms. <em>Annu Rev Neurosci</em>. 1990;13:475&ndash;511. <a href="https://doi.org/10.1146/annurev.ne.13.030190.002355">doi:10.1146/annurev.ne.13.030190.002355</a></li>
<li id="r32">Newell A, Simon H. The logic theory machine &mdash; a complex information processing system. <em>IRE Trans Inf Theory</em>. 1956;2(3):61&ndash;79. <a href="https://doi.org/10.1109/TIT.1956.1056797">doi:10.1109/TIT.1956.1056797</a></li>
<li id="r33">Weizenbaum J. ELIZA &mdash; a computer program for the study of natural language communication between man and machine. <em>Commun ACM</em>. 1966;9(1):36&ndash;45. <a href="https://doi.org/10.1145/365153.365168">doi:10.1145/365153.365168</a></li>
<li id="r34">Ferrucci D, Brown E, Chu-Carroll J, et al. Building Watson: an overview of the DeepQA project. <em>AI Magazine</em>. 2010;31(3):59&ndash;79. <a href="https://doi.org/10.1609/aimag.v31i3.2303">doi:10.1609/aimag.v31i3.2303</a></li>
<li id="r35">Vaswani A, Shazeer N, Parmar N, et al. Attention is all you need. <em>Advances in Neural Information Processing Systems 30 (NeurIPS 2017)</em>. <a href="https://doi.org/10.48550/arXiv.1706.03762">doi:10.48550/arXiv.1706.03762</a></li>
<li id="r36">Devlin J, Chang M-W, Lee K, Toutanova K. BERT: pre-training of deep bidirectional transformers for language understanding. <em>Proc NAACL-HLT 2019</em>. 2019:4171&ndash;4186. <a href="https://doi.org/10.18653/v1/N19-1423">doi:10.18653/v1/N19-1423</a></li>
<li id="r37">Chesney R, Citron D. Deep fakes: a looming challenge for privacy, democracy, and national security. <em>Calif Law Rev</em>. 2019;107(6):1753&ndash;1820. <a href="https://doi.org/10.15779/Z38RV0D15J">doi:10.15779/Z38RV0D15J</a></li>
<li id="r38">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R (incluye los datos originales de cáncer de mama de Wisconsin usados aquí). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r39">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r40">Therneau TM, Atkinson EJ. An Introduction to Recursive Partitioning Using the RPART Routines. Informe técnico de la Mayo Foundation; paquete de R <code>rpart</code>. <a href="https://doi.org/10.32614/CRAN.package.rpart">doi:10.32614/CRAN.package.rpart</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. &mdash; Capítulos 1&ndash;2 para las definiciones y la historia de esta unidad; la referencia estándar del campo.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. &mdash; Capítulo 1 para la visión estadística del reconocimiento de patrones. PDF gratuito en el sitio del autor.</li>
<li>Duda RO, Hart PE, Stork DG. <em>Pattern Classification</em>. 2nd ed. Wiley; 2001. &mdash; El texto clásico de reconocimiento de patrones; el Capítulo 1 define el pipeline de la Sección 4.2.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. &mdash; El complemento en R de todo el curso; PDF gratuito en statlearning.com.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. &mdash; La referencia para las funciones de modelado de R usadas a lo largo de esta edición.</li>
<li>Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25:44&ndash;56. &mdash; Una revisión legible de las aplicaciones médicas y de la evidencia que las respalda.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Documentación del paquete mlbench &mdash; <a href="https://cran.r-project.org/package=mlbench">cran.r-project.org/package=mlbench</a>; los datos de cáncer de mama y otros conjuntos de datos de referencia (benchmark) usados en este curso.</li>
<li>UCI Machine Learning Repository &mdash; <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; fuente original de los datos de cáncer de mama y de dígitos.</li>
<li>Biblioteca del Congreso Nacional de Chile, Ley Chile &mdash; <a href="https://www.bcn.cl/leychile">bcn.cl/leychile</a>; textos refundidos de las Leyes 19.628, 20.584 y 21.719.</li>
<li>FDA, Artificial Intelligence and Machine Learning in Software as a Medical Device &mdash; <a href="https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-and-machine-learning-software-medical-device">fda.gov</a>.</li>
<li>Guía de la OMS sobre ética y gobernanza de la IA para la salud &mdash; <a href="https://www.who.int/publications/i/item/9789240029200">who.int</a>.</li>
</ul>
